In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.random import default_rng
from scipy.linalg import expm
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEEDS=list(range(101,111))
N_TRAIN=5000
N_TEST=2000
TIMES=np.linspace(0,1.2,15)
KAPPA=1.0
NPHOT=2e4
SIGMA_DET=.002
SIGMA_PHASE=.01
print("Seeds:",SEEDS)

In [ ]:
def H_ep(theta,kappa=KAPPA):
    eps,d,dk,dw,loss=theta
    k=kappa*(1+dk); gamma=k+d
    return np.array([[dw+eps+1j*gamma-1j*loss,k],
                     [k,dw-eps-1j*gamma-1j*loss]],complex)

def propagate(theta,times=TIMES,psi0=None,kappa=KAPPA):
    if psi0 is None: psi0=np.array([1+0j,0+0j])
    H=H_ep(theta,kappa)
    return np.asarray([expm(-1j*H*t)@psi0 for t in times])

def observables(theta,times=TIMES,psi0=None,kappa=KAPPA):
    """
    Return both raw populations and normalized dynamical channels.

    Columns:
      0: log survival P = log(p1_raw+p2_raw)
      1: raw population p1_raw
      2: raw population p2_raw
      3: normalized population imbalance z
      4: normalized Re coherence
      5: normalized Im coherence
      6: first-component phase reference, phi = arg(psi_1)

    Keeping raw populations explicit is essential: p1/P and p2/P sum to one
    and therefore cannot carry survival/loss information by themselves.
    """
    psi=propagate(theta,times,psi0,kappa)
    p1=np.abs(psi[:,0])**2
    p2=np.abs(psi[:,1])**2
    P=np.maximum(p1+p2,1e-15)
    coh=np.conj(psi[:,0])*psi[:,1]
    # Phase-reference channel: phase of the first state component psi_1.
    # This is the channel used consistently by the simulation and Fisher model.
    phi=np.unwrap(np.angle(psi[:,0]))
    z=(p1-p2)/P
    cx=2*np.real(coh)/P
    cy=2*np.imag(coh)/P
    return np.column_stack([np.log(P),p1,p2,z,cx,cy,phi])

probe0=np.array([1+0j,0+0j])
probe1=np.array([1/np.sqrt(2),1/np.sqrt(2)],complex)


In [ ]:

def numerical_jacobian(func, theta, rel_step=1e-5, abs_step=1e-7):
    theta = np.asarray(theta, dtype=float)
    y0 = np.asarray(func(theta), dtype=float).ravel()
    J = np.empty((y0.size, theta.size), dtype=float)
    for j in range(theta.size):
        h = max(abs_step, rel_step * max(1.0, abs(theta[j])))
        tp = theta.copy(); tm = theta.copy()
        tp[j] += h; tm[j] -= h
        yp = np.asarray(func(tp), dtype=float).ravel()
        ym = np.asarray(func(tm), dtype=float).ravel()
        if yp.shape != y0.shape or ym.shape != y0.shape:
            raise ValueError("numerical_jacobian: function output shape changed.")
        J[:, j] = (yp - ym) / (2.0*h)
    return J



def channel_moments(theta, psi0, Nphot=NPHOT,
                    sigma_det=SIGMA_DET,
                    sigma_phase=SIGMA_PHASE,
                    kappa=KAPPA):
    o = observables(theta, TIMES, psi0, kappa)
    p1 = np.maximum(o[:,1], 1e-15)
    p2 = np.maximum(o[:,2], 1e-15)
    P = np.maximum(p1 + p2, 1e-15)

    mean = np.column_stack([
        np.log(P),
        o[:,3],
        o[:,4],
        o[:,5],
        o[:,6]
    ]).ravel()

    var_logP = 1.0/(Nphot*P) + sigma_det**2
    var_z = 4.0*p1*p2/(Nphot*np.maximum(P,1e-15)**3)
    var_c = np.full_like(P, sigma_det**2)
    var_phi = np.full_like(P, sigma_phase**2)

    # One diagonal variance vector per time/channel; first-order
    # Cov(logP,z) is zero for independent Poisson n1,n2.
    variances = np.column_stack([
        var_logP, var_z, var_c, var_c, var_phi
    ]).ravel()

    return mean, np.maximum(variances, 1e-18)


def fisher_from_reconstructed_channels(theta, psi0, Nphot=NPHOT,
                                       sigma_det=SIGMA_DET,
                                       sigma_phase=SIGMA_PHASE,
                                       kappa=KAPPA):
    def mean_fn(th):
        return channel_moments(
            th, psi0, Nphot=Nphot,
            sigma_det=sigma_det,
            sigma_phase=sigma_phase,
            kappa=kappa
        )[0]

    def var_fn(th):
        return channel_moments(
            th, psi0, Nphot=Nphot,
            sigma_det=sigma_det,
            sigma_phase=sigma_phase,
            kappa=kappa
        )[1]

    mean, var = channel_moments(
        theta, psi0, Nphot=Nphot,
        sigma_det=sigma_det,
        sigma_phase=sigma_phase,
        kappa=kappa
    )

    Jm = numerical_jacobian(mean_fn, theta)
    Jv = numerical_jacobian(var_fn, theta)

    inv_var = 1.0/np.maximum(var, 1e-18)

    # Mean contribution: J^T Sigma^{-1} J.
    F_mean = Jm.T @ (inv_var[:,None] * Jm)

    # Covariance contribution for diagonal Sigma(theta):
    # 1/2 * sum_k (d var_k/dtheta_a)(d var_k/dtheta_b)/var_k^2.
    F_cov = 0.5 * Jv.T @ ((inv_var**2)[:,None] * Jv)

    return F_mean + F_cov


def fisher(theta, Nphot=NPHOT, sigma_det=SIGMA_DET,
           include_phase=True):
    # The public identifiability audit uses the fixed probe pair.
    # Phase noise can be disabled for the corresponding baseline audit.
    phase_sigma = SIGMA_PHASE if include_phase else 1e12

    F = (
        fisher_from_reconstructed_channels(
            theta, probe0, Nphot=Nphot,
            sigma_det=sigma_det,
            sigma_phase=phase_sigma,
            kappa=KAPPA
        )
        +
        fisher_from_reconstructed_channels(
            theta, probe1, Nphot=Nphot,
            sigma_det=sigma_det,
            sigma_phase=phase_sigma,
            kappa=KAPPA
        )
    )
    return F


def fisher_grid(grid, include_phase):
    rows=[]
    for e in grid:
        for d in grid:
            th=np.array([e,d,.01,.01,.005])
            F=fisher(th,include_phase=include_phase)
            ev=np.linalg.eigvalsh((F+F.T)/2)
            ev=np.maximum(ev,0)
            rank=int(np.sum(ev>max(ev)*1e-8))
            rows.append([
                e,d,rank,ev[0],ev[-1],
                ev[-1]/max(ev[0],1e-30)
            ])
    return pd.DataFrame(
        rows,
        columns=[
            "epsilon","d_ep","rank",
            "lambda_min","lambda_max","condition"
        ]
    )

ep_grid=np.linspace(-.06,.06,13)
Fbase=fisher_grid(ep_grid,False)
Fphase=fisher_grid(ep_grid,True)

print("Baseline rank distribution:")
print(Fbase["rank"].value_counts().sort_index())
print("Phase-reference rank distribution:")
print(Fphase["rank"].value_counts().sort_index())


# 3. Fresh-probe measurement model

Every sample starts from a known probe state. The two probes provide ten time-resolved channels:

\[
[\log P,z,C_x,C_y,\phi]_{\psi_0}
\quad\text{and}\quad
[\log P,z,C_x,C_y,\phi]_{\psi_1}.
\]

Poisson fluctuations are applied to population counts and detector noise is added after reconstruction.

In [ ]:
def noisy_probe(theta,psi0,rng,Nphot=NPHOT,sigma_det=SIGMA_DET,
               sigma_phase=SIGMA_PHASE,kappa=KAPPA):
    """Generate the five measured channels from raw population counts.

    The Poisson counts use the RAW populations p1,p2, so their total
    n1+n2 contains genuine survival/loss information.  logP is derived
    from that same total count; it is not treated as an independent noisy
    measurement in the Fisher calculation, avoiding double counting.
    """
    o=observables(theta,TIMES,psi0,kappa)

    # IMPORTANT: columns 1 and 2 are raw populations, not normalized ones.
    p1=np.maximum(o[:,1],0.0)
    p2=np.maximum(o[:,2],0.0)

    n1=rng.poisson(Nphot*p1)
    n2=rng.poisson(Nphot*p2)

    q1=n1/Nphot
    q2=n2/Nphot
    P=np.maximum(q1+q2,1e-12)

    # Survival/loss channel from the total detected population.
    logP=np.log(P)+sigma_det*rng.normal(size=len(TIMES))

    # Conditional population imbalance from the same counts.
    z=(q1-q2)/P

    # Normalized coherence and phase channels.
    cx=o[:,4]+sigma_det*rng.normal(size=len(TIMES))
    cy=o[:,5]+sigma_det*rng.normal(size=len(TIMES))
    phi=o[:,6]+sigma_phase*rng.normal(size=len(TIMES))

    return np.column_stack([logP,z,cx,cy,phi])

def sample_theta(rng,full_drift=True,kappa_range=(.90,1.10)):
    eps=rng.uniform(-.06,.06)
    d=rng.uniform(-.12,.12) if full_drift else rng.uniform(-.06,.06)
    dk=rng.normal(0,.015)
    dw=rng.normal(0,.015)
    loss=abs(rng.normal(0,.004))
    kappa=rng.uniform(*kappa_range)
    return np.array([eps,d,dk,dw,loss]),kappa

def make_dataset(n,seed,kappa_range=(.90,1.10)):
    rng=default_rng(seed);X=[];Y=[];K=[]
    for _ in range(n):
        th,k=sample_theta(rng,True,kappa_range)
        a=noisy_probe(th,probe0,rng,kappa=k)
        b=noisy_probe(th,probe1,rng,kappa=k)
        X.append(np.column_stack([a,b]).ravel())
        Y.append(th[:2]);K.append(k)
    return np.asarray(X),np.asarray(Y),np.asarray(K)


In [ ]:
I2=np.eye(2,dtype=complex)
Xq=np.array([[0,1],[1,0]],complex)
Yq=np.array([[0,-1j],[1j,0]],complex)
Zq=np.diag([1,-1]).astype(complex)

def kronn(*ops):
    z=ops[0]
    for op in ops[1:]: z=np.kron(z,op)
    return z

def R(P,a):
    return np.cos(a/2)*I2-1j*np.sin(a/2)*P

def cnot(n,c,t):
    U=np.zeros((2**n,2**n),complex)
    for i in range(2**n):
        bits=[(i>>(n-1-j))&1 for j in range(n)]
        if bits[c]: bits[t]^=1
        j=0
        for b in bits:j=(j<<1)|b
        U[j,i]=1
    return U

ENT=cnot(3,2,0)@cnot(3,1,2)@cnot(3,0,1)

QOBS=[]
for q in range(3):
    for P in [Xq,Yq,Zq]:
        o=[I2]*3;o[q]=P;QOBS.append(kronn(*o))
for P in [Xq,Yq,Zq]:
    for pair in [(0,1),(1,2)]:
        o=[I2]*3;o[pair[0]]=P;o[pair[1]]=P;QOBS.append(kronn(*o))
QOBS += [kronn(Xq,Xq,Xq),kronn(Zq,Zq,Zq)]
Ostack=np.stack(QOBS)

def qrc_features(X,seed=1,noise=.02):
    r=default_rng(seed);n=X.shape[0];nt=len(TIMES);nin=10
    seq=X.reshape(n,nt,nin)
    scale=np.array([.8,1,.5,.5,np.pi]*2)
    W1=r.normal(0,.45,(3,nin));W2=r.normal(0,.30,(3,nin))
    feats=np.zeros((n,nt*len(QOBS)+len(QOBS)))
    rho=np.zeros((n,8,8),complex);rho[:,0,0]=1

    K0=np.array([[1,0],[0,np.sqrt(1-noise)]],complex)
    K1=np.array([[0,np.sqrt(noise)],[0,0]],complex)
    damp=[]
    for q in range(3):
        o0=[I2]*3;o1=[I2]*3;o0[q]=K0;o1[q]=K1
        damp.append((kronn(*o0),kronn(*o1)))

    for t in range(nt):
        u=seq[:,t,:]/scale;a=u@W1.T;b=u@W2.T
        ca=np.cos(a/2);sa=np.sin(a/2)
        Rx=np.zeros((n,2,2),complex);Rx[:,0,0]=ca[:,0];Rx[:,1,1]=ca[:,0]
        Rx[:,0,1]=-1j*sa[:,0];Rx[:,1,0]=-1j*sa[:,0]
        Ry=np.zeros((n,2,2),complex);Ry[:,0,0]=ca[:,1];Ry[:,1,1]=ca[:,1]
        Ry[:,0,1]=-sa[:,1];Ry[:,1,0]=sa[:,1]
        Rz=np.zeros((n,2,2),complex);Rz[:,0,0]=np.exp(-1j*a[:,2]/2);Rz[:,1,1]=np.exp(1j*a[:,2]/2)

        def bk(A,B,C):
            return np.einsum('nij,nkl,nmo->nikmjlo',A,B,C).reshape(n,8,8)

        U=bk(Rx,Ry,Rz)
        ca2=np.cos(b/2);sa2=np.sin(b/2)
        Zb=np.zeros((n,2,2),complex);Zb[:,0,0]=np.exp(-1j*b[:,0]/2);Zb[:,1,1]=np.exp(1j*b[:,0]/2)
        Xb=np.zeros((n,2,2),complex);Xb[:,0,0]=ca2[:,1];Xb[:,1,1]=ca2[:,1];Xb[:,0,1]=-1j*sa2[:,1];Xb[:,1,0]=-1j*sa2[:,1]
        Yb=np.zeros((n,2,2),complex);Yb[:,0,0]=ca2[:,2];Yb[:,1,1]=ca2[:,2];Yb[:,0,1]=-sa2[:,2];Yb[:,1,0]=sa2[:,2]
        U2=bk(Zb,Xb,Yb)

        Uall=U2@ENT
        Uall=Uall.reshape(n,8,8)
        rho=Uall@rho@np.transpose(Uall.conj(),(0,2,1))
        for A0,A1 in damp:
            rho=A0@rho@A0.conj().T+A1@rho@A1.conj().T

        f=np.real(np.einsum('nij,kji->nk',rho,Ostack))
        feats[:,t*len(QOBS):(t+1)*len(QOBS)]=f

    feats[:,-len(QOBS):]=f
    return feats

In [ ]:
def fit_ridge(Xtr,Ytr,Xte,alpha=1e-2):
    m=make_pipeline(StandardScaler(),Ridge(alpha=alpha))
    m.fit(Xtr,Ytr)
    return m,m.predict(Xte)

def metric_dict(Y,P):
    return dict(
        RMSE_eps=np.sqrt(mean_squared_error(Y[:,0],P[:,0])),
        RMSE_d=np.sqrt(mean_squared_error(Y[:,1],P[:,1])),
        MAE_eps=mean_absolute_error(Y[:,0],P[:,0]),
        MAE_d=mean_absolute_error(Y[:,1],P[:,1]),
        R2_eps=r2_score(Y[:,0],P[:,0]),
        R2_d=r2_score(Y[:,1],P[:,1])
    )

# One seed is used for model selection only; final benchmark uses independent seeds.
Xsel,Ysel,_=make_dataset(2500,4242)
rng=default_rng(4243);idx=rng.permutation(len(Xsel));tr=idx[:1800];va=idx[1800:]
candidates=[]
for qs in [1,2,3]:
    for damp in [.01,.02,.04]:
        A=qrc_features(Xsel[tr],qs,damp);B=qrc_features(Xsel[va],qs,damp)
        for alpha in [1e-3,1e-2,1e-1]:
            m,p=fit_ridge(A,Ysel[tr],B,alpha)
            mm=metric_dict(Ysel[va],p)
            candidates.append((qs,damp,alpha,mm["RMSE_eps"]+mm["RMSE_d"]))
best=min(candidates,key=lambda x:x[-1])
QML_SEED,QML_DAMP,QML_ALPHA=best[:3]
print("Selected on validation only:",best)

In [ ]:
benchmark=[]
for seed in SEEDS:
    Xtr,Ytr,_=make_dataset(N_TRAIN,seed)
    Xte,Yte,_=make_dataset(N_TEST,seed+10000)

    mr,pr=fit_ridge(Xtr,Ytr,Xte,QML_ALPHA)
    benchmark.append({"seed":seed,"model":"ridge",**metric_dict(Yte,pr)})

    # Classical reservoir uses the same number of input channels and a fixed 120-node state.
    def classical_features(X,seed):
        n=len(X);nt=len(TIMES);nin=10
        seq=X.reshape(n,nt,nin);r=default_rng(seed)
        W=r.normal(size=(120,120));W*=.85/max(abs(np.linalg.eigvals(W)))
        Win=r.normal(scale=.7/np.sqrt(nin),size=(120,nin))
        s=np.zeros((n,120))
        for t in range(nt): s=np.tanh(s@W.T+seq[:,t,:]@Win.T)
        return s
    Rtr=classical_features(Xtr,seed);Rte=classical_features(Xte,seed)
    mc,pc=fit_ridge(Rtr,Ytr,Rte,QML_ALPHA)
    benchmark.append({"seed":seed,"model":"classical_reservoir",**metric_dict(Yte,pc)})

    Qtr=qrc_features(Xtr,QML_SEED,QML_DAMP);Qte=qrc_features(Xte,QML_SEED,QML_DAMP)
    mq,pq=fit_ridge(Qtr,Ytr,Qte,QML_ALPHA)
    benchmark.append({"seed":seed,"model":"QML_reservoir",**metric_dict(Yte,pq)})

benchmark_df=pd.DataFrame(benchmark)
display(benchmark_df.groupby("model")[["RMSE_eps","RMSE_d","MAE_eps","MAE_d","R2_eps","R2_d"]].agg(["mean","std"]))

In [ ]:
Xid,Yid,_=make_dataset(5000,555,(.90,1.10))
Xood,Yood,_=make_dataset(2000,556,(1.20,1.35))
ood=[]

for seed in SEEDS:
    mr,pr=fit_ridge(Xid,Yid,Xood,QML_ALPHA)
    ood.append({"seed":seed,"model":"ridge",**metric_dict(Yood,pr)})

    Rtr=classical_features(Xid,seed);Roo=classical_features(Xood,seed)
    mc,pc=fit_ridge(Rtr,Yid,Roo,QML_ALPHA)
    ood.append({"seed":seed,"model":"classical_reservoir",**metric_dict(Yood,pc)})

    Qtr=qrc_features(Xid,QML_SEED,QML_DAMP);Qoo=qrc_features(Xood,QML_SEED,QML_DAMP)
    mq,pq=fit_ridge(Qtr,Yid,Qoo,QML_ALPHA)
    ood.append({"seed":seed,"model":"QML_reservoir",**metric_dict(Yood,pq)})

ood_df=pd.DataFrame(ood)
display(ood_df.groupby("model")[["RMSE_eps","RMSE_d","R2_eps","R2_d"]].agg(["mean","std"]))

In [ ]:
rng=default_rng(700)
Xd=[];Yd=[]
for _ in range(6000):
    th,k=sample_theta(rng,True,(.90,1.10))
    a=noisy_probe(th,probe0,rng,kappa=k);b=noisy_probe(th,probe1,rng,kappa=k)
    Xd.append(np.column_stack([a,b]).ravel());Yd.append(th[:2])
Xd=np.asarray(Xd);Yd=np.asarray(Yd)

Qd=qrc_features(Xd,QML_SEED,QML_DAMP)
dynamic_model,_=fit_ridge(Qd,Yd,Qd,QML_ALPHA)

T=np.arange(0,8,.08);eps_true=.025
d_true=.075*np.sin(2*np.pi*.11*T)+.025*np.sin(2*np.pi*.29*T)
rng=default_rng(702);est=[]
for dnow in d_true:
    th=np.array([eps_true,dnow,.006,.004,.003])
    a=noisy_probe(th,probe0,rng,kappa=1.0);b=noisy_probe(th,probe1,rng,kappa=1.0)
    x=np.column_stack([a,b]).ravel()[None,:]
    est.append(dynamic_model.predict(qrc_features(x,QML_SEED,QML_DAMP))[0])
est=np.asarray(est)
print("Dynamic RMSE d_EP:",np.sqrt(np.mean((est[:,1]-d_true)**2)))

# 9. Bias-aware closed-loop controller

The controller has two logically separate tasks:

1. estimate \(d_{\rm EP}\) for feedback;
2. estimate \(\epsilon\) for sensing.

The controller is not allowed to use the sensing estimate as its feedback signal.

A full-range drift estimator is trained over \(d_{\rm EP}\in[-0.12,0.12]\).

In [ ]:
# Controller training — full drift range with nuisance domain randomization
rng=default_rng(800);Xc=[];Yc=[]
for _ in range(8000):
    th,k=sample_theta(rng,True,(.90,1.10))
    a=noisy_probe(th,probe0,rng,kappa=k);b=noisy_probe(th,probe1,rng,kappa=k)
    Xc.append(np.column_stack([a,b]).ravel());Yc.append(th[:2])
Xc=np.asarray(Xc);Yc=np.asarray(Yc)
Qc=qrc_features(Xc,QML_SEED,QML_DAMP)
ctrl_model,_=fit_ridge(Qc,Yc,Qc,QML_ALPHA)

def controller_run(Kp,mode="qml",seed=900,steps=100):
    rng=default_rng(seed);k=1.;gamma=k+.10;eps=.025;hist=[]
    for n in range(steps):
        env=.10+.0002*n+.015*np.sin(.12*n)
        gamma+=.18*(env-(gamma-k));d=gamma-k
        if mode=="none": d_hat=0.
        elif mode=="oracle": d_hat=d
        else:
            th=np.array([eps,d,.006,.004,.003])
            a=noisy_probe(th,probe0,rng,kappa=k);b=noisy_probe(th,probe1,rng,kappa=k)
            x=np.column_stack([a,b]).ravel()[None,:]
            d_hat=ctrl_model.predict(qrc_features(x,QML_SEED,QML_DAMP))[0,1]
        gamma-=Kp*d_hat
        hist.append([n,d,d_hat])
    return np.asarray(hist)

KPS=np.linspace(.02,1.20,25);rows=[]
for mode in ["oracle","qml"]:
    for kp in KPS:
        finals=[];maxima=[]
        for s in range(5):
            h=controller_run(kp,mode,1000+s);finals.append(abs(np.mean(h[-10:,1])));maxima.append(np.max(abs(h[:,1])))
        rows.append([mode,kp,np.mean(finals),np.max(maxima)])
stability_df=pd.DataFrame(rows,columns=["mode","Kp","final_abs_d","max_abs_d"])
display(stability_df)

In [ ]:
def sensing_bias_experiment(Kp=.35,seed=1200,steps=100):
    rng=default_rng(seed);k=1.;gamma=k+.10;eps=.025;rows=[]
    for n in range(steps):
        env=.10+.0002*n+.015*np.sin(.12*n)
        gamma+=.18*(env-(gamma-k));d=gamma-k
        th=np.array([eps,d,.006,.004,.003])
        a=noisy_probe(th,probe0,rng,kappa=k);b=noisy_probe(th,probe1,rng,kappa=k)
        x=np.column_stack([a,b]).ravel()[None,:]
        pred=ctrl_model.predict(qrc_features(x,QML_SEED,QML_DAMP))[0]
        eps_hat,d_hat=pred
        gamma-=Kp*d_hat
        rows.append([n,d,eps_hat,eps_hat-eps])
    out=pd.DataFrame(rows,columns=["cycle","d_true","eps_hat","eps_bias"])
    out["phase"]=np.where(out.cycle<10,"before",np.where(out.cycle<40,"during","after"))
    return out

bias_df=sensing_bias_experiment()
bias_summary=bias_df.groupby("phase")[["d_true","eps_bias"]].agg(["mean","std"])
display(bias_summary)

In [ ]:
benchmark_df.to_csv("V2_2_QML_10seed_benchmark.csv",index=False)
ood_df.to_csv("V2_2_QML_strict_OOD.csv",index=False)
bias_summary.to_csv("V2_2_sensing_bias_summary.csv")
Fbase.to_csv("V2_2_Fisher_baseline_map.csv",index=False)
Fphase.to_csv("V2_2_Fisher_phase_map.csv",index=False)

print("Saved publication audit tables.")

In [ ]:
from sklearn.model_selection import train_test_split
from scipy.stats import ttest_rel
import numpy as np, pandas as pd

STRICT_SEEDS=list(range(101,111))

def paired_bootstrap(x, y, B=10000, seed=12345):
    x=np.asarray(x); y=np.asarray(y)
    d=x-y
    rng=np.random.default_rng(seed)
    idx=rng.integers(0,len(d),(B,len(d)))
    means=d[idx].mean(axis=1)
    return float(d.mean()), np.quantile(means,[.025,.975])

def paired_summary(df, metric, model_a="QML_reservoir", model_b="classical_reservoir"):
    a=df[df.model==model_a].set_index("seed")[metric]
    b=df[df.model==model_b].set_index("seed")[metric]
    common=a.index.intersection(b.index)
    diff=a.loc[common].values-b.loc[common].values
    stat,p=ttest_rel(a.loc[common],b.loc[common])
    mean,ci=paired_bootstrap(a.loc[common].values,b.loc[common].values)
    return {"metric":metric,"QML_minus_classical_mean":mean,
            "95CI_low":ci[0],"95CI_high":ci[1],"paired_t":stat,"p":p}

print("Strict statistical utilities loaded.")

In [ ]:
def strict_split_dataset(seed, ntr=5000, nva=1500, nte=2000):
    Xtr,Ytr,Ktr=make_dataset(ntr,seed,(.90,1.10))
    Xva,Yva,Kva=make_dataset(nva,seed+1000,(.90,1.10))
    Xte,Yte,Kte=make_dataset(nte,seed+2000,(.90,1.10))
    return Xtr,Ytr,Xva,Yva,Xte,Yte,Ktr,Kva,Kte

def select_qml_on_validation(Xtr,Ytr,Xva,Yva,seed):
    # Keep the candidate set deliberately small and fixed in advance.
    candidates=[]
    for damp in [.01,.02,.04]:
        for alpha in [1e-3,1e-2,1e-1]:
            A=qrc_features(Xtr,seed,damp)
            B=qrc_features(Xva,seed,damp)
            m,p=fit_ridge(A,Ytr,B,alpha)
            mm=metric_dict(Yva,p)
            candidates.append((mm["RMSE_eps"]+mm["RMSE_d"],damp,alpha))
    _,damp,alpha=min(candidates,key=lambda z:z[0])
    return damp,alpha

strict_rows=[]
for seed in STRICT_SEEDS:
    Xtr,Ytr,Xva,Yva,Xte,Yte,*_=strict_split_dataset(seed)
    damp,alpha=select_qml_on_validation(Xtr,Ytr,Xva,Yva,seed)

    # Ridge
    mr,pr=fit_ridge(Xtr,Ytr,Xte,alpha)
    strict_rows.append({"seed":seed,"model":"ridge",**metric_dict(Yte,pr)})

    # Classical reservoir: independently seeded for every seed.
    Rtr=classical_features(Xtr,seed)
    Rva=classical_features(Xva,seed)
    Rte=classical_features(Xte,seed)
    mc,pc=fit_ridge(Rtr,Ytr,Rte,alpha)
    strict_rows.append({"seed":seed,"model":"classical_reservoir",**metric_dict(Yte,pc)})

    # QML: independently seeded for every seed.
    Qtr=qrc_features(Xtr,seed,damp)
    Qva=qrc_features(Xva,seed,damp)
    Qte=qrc_features(Xte,seed,damp)
    mq,pq=fit_ridge(Qtr,Ytr,Qte,alpha)
    strict_rows.append({"seed":seed,"model":"QML_reservoir",**metric_dict(Yte,pq)})

strict_benchmark=pd.DataFrame(strict_rows)
display(strict_benchmark.groupby("model")[["RMSE_eps","RMSE_d","MAE_eps","MAE_d","R2_eps","R2_d"]].agg(["mean","std"]))
display(pd.DataFrame([
    paired_summary(strict_benchmark,"RMSE_eps"),
    paired_summary(strict_benchmark,"RMSE_d")
]))

In [ ]:
ood_rows=[]
for seed in STRICT_SEEDS:
    Xtr,Ytr,_=make_dataset(5000,seed,(.90,1.10))
    Xva,Yva,_=make_dataset(1500,seed+1000,(.90,1.10))
    Xood,Yood,_=make_dataset(2000,seed+2000,(1.20,1.35))

    damp,alpha=select_qml_on_validation(Xtr,Ytr,Xva,Yva,seed)

    mr,pr=fit_ridge(Xtr,Ytr,Xood,alpha)
    ood_rows.append({"seed":seed,"model":"ridge",**metric_dict(Yood,pr)})

    Rtr=classical_features(Xtr,seed); Roo=classical_features(Xood,seed)
    mc,pc=fit_ridge(Rtr,Ytr,Roo,alpha)
    ood_rows.append({"seed":seed,"model":"classical_reservoir",**metric_dict(Yood,pc)})

    Qtr=qrc_features(Xtr,seed,damp); Qoo=qrc_features(Xood,seed,damp)
    mq,pq=fit_ridge(Qtr,Ytr,Qoo,alpha)
    ood_rows.append({"seed":seed,"model":"QML_reservoir",**metric_dict(Yood,pq)})

strict_ood=pd.DataFrame(ood_rows)
display(strict_ood.groupby("model")[["RMSE_eps","RMSE_d","R2_eps","R2_d"]].agg(["mean","std"]))
display(pd.DataFrame([
    paired_summary(strict_ood,"RMSE_eps"),
    paired_summary(strict_ood,"RMSE_d")
]))

In [ ]:
def make_dynamic_trajectory(seed, steps=100):
    rng=np.random.default_rng(seed)
    T=np.arange(steps)*.08
    # Independent smooth drift trajectory.
    d=.09*np.sin(.09*T+rng.uniform(0,2*np.pi)) + .025*np.sin(.31*T+rng.uniform(0,2*np.pi))
    eps=.025+0.005*np.sin(.05*T+rng.uniform(0,2*np.pi))
    return T,eps,d

def dynamic_dataset(seed, ntraj=100, steps=100):
    rng=np.random.default_rng(seed); X=[];Y=[];traj=[]
    for j in range(ntraj):
        T,e,d=make_dynamic_trajectory(seed+j,steps)
        for tt,ee,dd in zip(T,e,d):
            th=np.array([ee,dd,.006,.004,.003])
            a=noisy_probe(th,probe0,rng,kappa=1.0)
            b=noisy_probe(th,probe1,rng,kappa=1.0)
            X.append(np.column_stack([a,b]).ravel())
            Y.append([ee,dd]);traj.append(j)
    return np.asarray(X),np.asarray(Y),np.asarray(traj)

dyn_train_X,dyn_train_Y,dyn_train_id=dynamic_dataset(20000,60,80)
dyn_test_X,dyn_test_Y,dyn_test_id=dynamic_dataset(30000,30,120)

dyn_qtr=qrc_features(dyn_train_X,7,.02)
dyn_qte=qrc_features(dyn_test_X,7,.02)
dyn_model,_=fit_ridge(dyn_qtr,dyn_train_Y,dyn_qte,1e-2)
dyn_pred=dyn_model.predict(dyn_qte)

dynamic_metrics={
    "RMSE_eps":np.sqrt(np.mean((dyn_pred[:,0]-dyn_test_Y[:,0])**2)),
    "RMSE_d":np.sqrt(np.mean((dyn_pred[:,1]-dyn_test_Y[:,1])**2)),
    "MAE_eps":np.mean(np.abs(dyn_pred[:,0]-dyn_test_Y[:,0])),
    "MAE_d":np.mean(np.abs(dyn_pred[:,1]-dyn_test_Y[:,1])),
    "MaxAbs_d":np.max(np.abs(dyn_pred[:,1]-dyn_test_Y[:,1]))
}
print(dynamic_metrics)

In [ ]:
def controller_run_strict(Kp,mode,seed,steps=120):
    rng=np.random.default_rng(seed)
    k=1.;gamma=k+.08;eps=.025
    hist=[]
    for n in range(steps):
        env=.08+.0003*n+.018*np.sin(.11*n+rng.uniform(-.2,.2))
        gamma += .18*(env-(gamma-k))
        d=gamma-k

        if mode=="none":
            d_hat=0.
        elif mode=="oracle":
            d_hat=d
        else:
            th=np.array([eps,d,.006,.004,.003])
            a=noisy_probe(th,probe0,rng,kappa=k)
            b=noisy_probe(th,probe1,rng,kappa=k)
            x=np.column_stack([a,b]).ravel()[None,:]
            # strict controller model is passed globally below
            d_hat=strict_ctrl.predict(
                qrc_features(x,strict_ctrl_seed,strict_ctrl_damp)
            )[0,1]

        gamma -= Kp*d_hat
        hist.append([n,d,d_hat])
    return np.asarray(hist)

# Train controller on disjoint trajectory family.
cx,cy,_=dynamic_dataset(40000,100,60)
strict_ctrl_seed=17
strict_ctrl_damp=.02
strict_ctrl,_=fit_ridge(
    qrc_features(cx,strict_ctrl_seed,strict_ctrl_damp),
    cy,
    qrc_features(cx,strict_ctrl_seed,strict_ctrl_damp),
    1e-2
)

# Validation gain selection.
gain_candidates=np.linspace(.05,.80,16)
val_scores=[]
for kp in gain_candidates:
    vals=[]
    for s in range(10):
        h=controller_run_strict(kp,"qml",50000+s,100)
        vals.append(np.mean(np.abs(h[-20:,1])))
    val_scores.append((np.mean(vals),kp))
best_kp=min(val_scores)[1]
print("Validation-selected Kp:",best_kp)

# Independent test trajectories.
controller_rows=[]
for s in range(20):
    for mode in ["none","oracle","qml"]:
        h=controller_run_strict(best_kp,mode,60000+s,120)
        controller_rows.append({
            "seed":s,"mode":mode,
            "final_abs_d":np.mean(np.abs(h[-20:,1])),
            "RMSE_d":np.sqrt(np.mean(h[:,1]**2)),
            "max_abs_d":np.max(np.abs(h[:,1]))
        })

strict_controller=pd.DataFrame(controller_rows)
display(strict_controller.groupby("mode")[["final_abs_d","RMSE_d","max_abs_d"]].agg(["mean","std"]))

In [ ]:
def bias_run(seed,Kp,steps=120):
    rng=np.random.default_rng(seed);k=1.;gamma=k+.08;eps=.025;rows=[]
    for n in range(steps):
        env=.08+.0003*n+.018*np.sin(.11*n+rng.uniform(-.2,.2))
        gamma += .18*(env-(gamma-k));d=gamma-k
        th=np.array([eps,d,.006,.004,.003])
        a=noisy_probe(th,probe0,rng,kappa=k);b=noisy_probe(th,probe1,rng,kappa=k)
        x=np.column_stack([a,b]).ravel()[None,:]
        pred=strict_ctrl.predict(qrc_features(x,strict_ctrl_seed,strict_ctrl_damp))[0]
        gamma -= Kp*pred[1]
        rows.append([n,d,pred[0],pred[0]-eps])
    df=pd.DataFrame(rows,columns=["cycle","d_true","eps_hat","eps_bias"])
    df["phase"]=np.select([df.cycle<15,df.cycle<55],["before","during"],default="after")
    return df

allbias=pd.concat([bias_run(70000+s,best_kp) for s in range(30)],ignore_index=True)

def bootstrap_mean_ci(x,B=10000,seed=777):
    x=np.asarray(x); rng=np.random.default_rng(seed)
    means=x[rng.integers(0,len(x),(B,len(x)))].mean(axis=1)
    return float(np.mean(x)),tuple(np.quantile(means,[.025,.975]))

bias_rows=[]
for phase,g in allbias.groupby("phase"):
    m,ci=bootstrap_mean_ci(g.eps_bias.values)
    bias_rows.append([phase,m,g.eps_bias.std(ddof=1),ci[0],ci[1]])
strict_bias=pd.DataFrame(bias_rows,columns=["phase","mean_bias","std","CI_low","CI_high"])
display(strict_bias)

In [ ]:
# Save all strict-audit tables.
strict_benchmark.to_csv("V2_3_strict_10seed_benchmark.csv",index=False)
strict_ood.to_csv("V2_3_strict_unseen_device.csv",index=False)
strict_controller.to_csv("V2_3_strict_controller_test.csv",index=False)
strict_bias.to_csv("V2_3_strict_sensing_bias.csv",index=False)

print("Strict V2.3 audit tables saved.")

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

QML_ENSEMBLE_SEEDS=[11,23,37,41]
# Training/validation device domains. For each sample, one of these domains
# is selected and kappa is drawn within that domain. This is true sample-level
# domain randomization rather than selecting one range for an entire dataset.
DOMAIN_RANGES=[(.88,1.12),(.85,1.15),(.80,1.20)]

def fit_scaler_v24(X):
    mu=X.mean(axis=0)
    sd=np.where(X.std(axis=0)<1e-12,1.0,X.std(axis=0))
    return mu,sd

def apply_scaler_v24(X,mu,sd):
    return (X-mu)/sd

def qml_ensemble_features_v24(X,seeds,damp):
    return np.concatenate([qrc_features(X,s,damp) for s in seeds],axis=1)

def metrics_v24(y,p):
    return {
        "RMSE_eps":float(np.sqrt(np.mean((p[:,0]-y[:,0])**2))),
        "RMSE_d":float(np.sqrt(np.mean((p[:,1]-y[:,1])**2))),
        "MAE_eps":float(np.mean(np.abs(p[:,0]-y[:,0]))),
        "MAE_d":float(np.mean(np.abs(p[:,1]-y[:,1]))),
        "R2_eps":float(r2_score(y[:,0],p[:,0])),
        "R2_d":float(r2_score(y[:,1],p[:,1]))
    }

def domain_dataset_v24(n,seed):
    rng=np.random.default_rng(seed)
    X,Y,K=[],[],[]
    for _ in range(n):
        # Select the domain independently for each sample.
        lo,hi=DOMAIN_RANGES[rng.integers(len(DOMAIN_RANGES))]
        th,k=sample_theta(rng,True,(lo,hi))
        a=noisy_probe(th,probe0,rng,kappa=k)
        b=noisy_probe(th,probe1,rng,kappa=k)
        X.append(np.column_stack([a,b]).ravel())
        Y.append(th[:2])
        K.append(k)
    return np.asarray(X),np.asarray(Y),np.asarray(K)


In [ ]:
# Generate one fixed train/validation pair for all ensemble sizes.
# This isolates the effect of M from dataset-to-dataset variation.
Xtr_val,Ytr_val,Ktr_val=domain_dataset_v24(4000,8001)
Xva_val,Yva_val,Kva_val=domain_dataset_v24(1500,9001)

val_rows=[]
for M in [1,2,4]:
    seeds=QML_ENSEMBLE_SEEDS[:M]
    A=qml_ensemble_features_v24(Xtr_val,seeds,.02)
    V=qml_ensemble_features_v24(Xva_val,seeds,.02)
    mu,sd=fit_scaler_v24(A)
    A=apply_scaler_v24(A,mu,sd); V=apply_scaler_v24(V,mu,sd)
    model=Ridge(alpha=1e-2).fit(A,Ytr_val)
    val_rows.append({"M":M,**metrics_v24(Yva_val,model.predict(V))})

v24_val=pd.DataFrame(val_rows)
display(v24_val)
v24_val["selection_score"]=v24_val.RMSE_eps+v24_val.RMSE_d
best_M=int(v24_val.sort_values("selection_score").iloc[0].M)
BEST_QML_SEEDS=QML_ENSEMBLE_SEEDS[:best_M]
print("Validation-selected M =",best_M)
print("All M values use identical Xtr/Ytr and Xva/Yva datasets.")


In [ ]:
rows=[]
for seed in range(101,111):
    # Training/validation use sample-level mixture domain randomization.
    Xtr,Ytr,_=domain_dataset_v24(5000,10000+seed)
    Xva,Yva,_=domain_dataset_v24(1500,11000+seed)
    Xte,Yte,_=make_dataset(2000,12000+seed,(1.22,1.38))

    A=qml_ensemble_features_v24(Xtr,BEST_QML_SEEDS,.02)
    V=qml_ensemble_features_v24(Xva,BEST_QML_SEEDS,.02)
    T=qml_ensemble_features_v24(Xte,BEST_QML_SEEDS,.02)

    mu,sd=fit_scaler_v24(A)
    A=apply_scaler_v24(A,mu,sd); V=apply_scaler_v24(V,mu,sd); T=apply_scaler_v24(T,mu,sd)

    candidates=[1e-4,1e-3,1e-2,1e-1]
    scores=[]
    for a in candidates:
        m=Ridge(alpha=a).fit(A,Ytr)
        scores.append((np.sqrt(np.mean((m.predict(V)-Yva)**2)),a))
    alpha=min(scores)[1]
    m=Ridge(alpha=alpha).fit(A,Ytr)
    rows.append({"seed":seed,"model":"QML_ensemble",**metrics_v24(Yte,m.predict(T))})

v24_ood=pd.DataFrame(rows)
display(v24_ood.groupby("model")[["RMSE_eps","RMSE_d","R2_eps","R2_d"]].agg(["mean","std"]))

In [ ]:
dyn_train_X,dyn_train_Y,_=dynamic_dataset(90000,100,80)
dyn_test_X,dyn_test_Y,_=dynamic_dataset(91000,40,120)

A=qml_ensemble_features_v24(dyn_train_X,BEST_QML_SEEDS,.02)
T=qml_ensemble_features_v24(dyn_test_X,BEST_QML_SEEDS,.02)
mu,sd=fit_scaler_v24(A)
A=apply_scaler_v24(A,mu,sd); T=apply_scaler_v24(T,mu,sd)

dyn_model=Ridge(alpha=1e-2).fit(A,dyn_train_Y)
dyn_pred=dyn_model.predict(T)
print(metrics_v24(dyn_test_Y,dyn_pred))

In [ ]:
ctrl_train_X,ctrl_train_Y,_=dynamic_dataset(93000,120,60)
ctrl_val_X,ctrl_val_Y,_=dynamic_dataset(94000,30,60)

A=qml_ensemble_features_v24(ctrl_train_X,BEST_QML_SEEDS,.02)
V=qml_ensemble_features_v24(ctrl_val_X,BEST_QML_SEEDS,.02)
mu,sd=fit_scaler_v24(A)
A=apply_scaler_v24(A,mu,sd); V=apply_scaler_v24(V,mu,sd)

controller_model=Ridge(alpha=1e-2).fit(A,ctrl_train_Y)
print("Controller model fitted only on the controller-training trajectory family.")

In [ ]:

# EXPERIMENT 1 — Measurement-cost analysis

import time
from scipy.linalg import eigvals
from sklearn.metrics import mean_squared_error


ACQ_TIME_PER_PROBE_POINT_MS = 1.0
SPECTRAL_POINT_TIME_MS = 1.0
SPECTRAL_POINTS = 41
SPECTRAL_REPEATS = 1

RESERVOIR_PROBES = 2
RESERVOIR_TIME_POINTS = len(TIMES)
RESERVOIR_POPULATION_READOUTS = RESERVOIR_PROBES * RESERVOIR_TIME_POINTS * 2
RESERVOIR_ACQUISITION_RECORDS = RESERVOIR_PROBES * RESERVOIR_TIME_POINTS
SPECTRAL_ACQUISITION_RECORDS = SPECTRAL_POINTS * SPECTRAL_REPEATS

cost_rng = np.random.default_rng(2401)
cost_theta = np.array([.025, .075, .006, .004, .003])
cost_kappa = 1.0

def repeated_spectral_scan(theta, n_points=SPECTRAL_POINTS, repeats=1):
    """Reference spectral recalibration: repeated eigenvalue scans."""
    vals=[]
    scan_axis=np.linspace(-0.08,0.08,n_points)
    for _ in range(repeats):
        for x in scan_axis:
            th=theta.copy(); th[0]=x
            vals.append(np.linalg.eigvals(H_ep(th,kappa=cost_kappa)))
    return np.asarray(vals)

X_cost=[]
for _ in range(32):
    a=noisy_probe(cost_theta,probe0,cost_rng,kappa=cost_kappa)
    b=noisy_probe(cost_theta,probe1,cost_rng,kappa=cost_kappa)
    X_cost.append(np.column_stack([a,b]).ravel())
X_cost=np.asarray(X_cost)

t0=time.perf_counter()
_=qml_ensemble_features_v24(X_cost,BEST_QML_SEEDS,.02)
reservoir_compute_s=time.perf_counter()-t0

t0=time.perf_counter()
for _ in range(32):
    _=repeated_spectral_scan(cost_theta)
spectral_compute_s=time.perf_counter()-t0

cost_rows=[
    {
        "method":"reservoir_self_calibration",
        "probe_or_scan_records":RESERVOIR_ACQUISITION_RECORDS,
        "population_readouts":RESERVOIR_POPULATION_READOUTS,
        "assumed_acquisition_time_ms":RESERVOIR_ACQUISITION_RECORDS*ACQ_TIME_PER_PROBE_POINT_MS,
        "compute_time_s_32_samples":reservoir_compute_s,
        "acquisition_time_basis":"assumed 1 ms per probe/time record; not hardware-measured",
        "notes":"2 fresh probes x 15 time points; multiple channels are reconstructed from the same acquisition."
    },
    {
        "method":"repeated_spectral_scan",
        "probe_or_scan_records":SPECTRAL_ACQUISITION_RECORDS,
        "population_readouts":np.nan,
        "assumed_acquisition_time_ms":SPECTRAL_ACQUISITION_RECORDS*SPECTRAL_POINT_TIME_MS,
        "compute_time_s_32_samples":spectral_compute_s,
        "acquisition_time_basis":"assumed 1 ms per spectral point; not hardware-measured",
        "notes":"41-point scan with eigendecomposition at each spectral point."
    }
]
measurement_cost_df=pd.DataFrame(cost_rows)
display(measurement_cost_df)

print("Acquisition-record ratio (spectral / reservoir):",
      SPECTRAL_ACQUISITION_RECORDS/RESERVOIR_ACQUISITION_RECORDS)
print("Measured compute-time ratio (spectral / reservoir):",
      spectral_compute_s/max(reservoir_compute_s,1e-12))
print("IMPORTANT: acquisition times above are assumed/modelled, not hardware measurements.")

measurement_cost_df.to_csv("V2_4_measurement_cost_analysis.csv",index=False)


In [ ]:

# EXPERIMENT 2 — Probe optimization using Fisher determinant / CRLB

from scipy.optimize import differential_evolution

TARGET_IDX=[0,1]
# Fisher/CRLB parameterization: [epsilon, d_EP, delta_kappa, delta_omega, loss, kappa].
# kappa is now explicitly treated as an unknown device parameter (nuisance).
NUISANCE_IDX=[2,3,4,5]
FISHER_KAPPA_GRID=np.linspace(.90,1.10,5)

def probe_from_angles(alpha,beta):
    return np.array([
        np.cos(alpha/2.0),
        np.exp(1j*beta)*np.sin(alpha/2.0)
    ],dtype=complex)

def fisher_for_probe_pair(theta,psi_a,psi_b,nphot=NPHOT,
                          sigma_det=SIGMA_DET,sigma_phase=SIGMA_PHASE,
                          kappa_value=KAPPA):
    """Five-parameter Fisher matrix with kappa held fixed.

    This version is retained for diagnostics where the device calibration
    value is assumed known. The CRLB/probe-design calculation below uses
    `fisher_for_probe_pair_with_kappa_nuisance()` instead.
    """
    return (
        fisher_from_reconstructed_channels(
            theta, psi_a, Nphot=nphot,
            sigma_det=sigma_det,
            sigma_phase=sigma_phase,
            kappa=kappa_value
        )
        +
        fisher_from_reconstructed_channels(
            theta, psi_b, Nphot=nphot,
            sigma_det=sigma_det,
            sigma_phase=sigma_phase,
            kappa=kappa_value
        )
    )

def fisher_for_probe_pair_with_kappa_nuisance(theta,psi_a,psi_b,nphot=NPHOT,
                                              sigma_det=SIGMA_DET,
                                              sigma_phase=SIGMA_PHASE,
                                              kappa_value=KAPPA):
    """Six-parameter Fisher matrix with kappa explicitly included.

    Parameter vector is [epsilon, d_EP, delta_kappa, delta_omega, loss, kappa].
    The first five coordinates are exactly the theta vector used elsewhere;
    kappa is appended as a nuisance parameter and is differentiated numerically
    through the full measurement model.
    """
    x0=np.concatenate([np.asarray(theta,dtype=float),[float(kappa_value)]])

    def moments_aug(x,psi0):
        th=np.asarray(x[:5],dtype=float)
        kap=float(x[5])
        return channel_moments(
            th,psi0,Nphot=nphot,
            sigma_det=sigma_det,sigma_phase=sigma_phase,kappa=kap
        )

    def one_probe_fisher(psi0):
        mean,var=moments_aug(x0,psi0)
        Jm=numerical_jacobian(lambda x:moments_aug(x,psi0)[0],x0)
        Jv=numerical_jacobian(lambda x:moments_aug(x,psi0)[1],x0)
        inv_var=1.0/np.maximum(var,1e-18)
        F_mean=Jm.T @ (inv_var[:,None]*Jm)
        F_cov=0.5*Jv.T @ ((inv_var**2)[:,None] * Jv)
        return F_mean+F_cov

    return one_probe_fisher(psi_a)+one_probe_fisher(psi_b)

def effective_target_fisher(F,target=TARGET_IDX,nuisance=NUISANCE_IDX):
    Ft=F[np.ix_(target,target)]
    if len(nuisance)==0:
        return Ft
    Fn=F[np.ix_(nuisance,nuisance)]
    Ftn=F[np.ix_(target,nuisance)]
    return Ft-Ftn@np.linalg.pinv(Fn,rcond=1e-10)@Ftn.T

probe_opt_theta=np.array([.025,.075,.006,.004,.003])
probe_design_points=[
    probe_opt_theta,
    np.array([0.0,.05,.006,.004,.003]),
    np.array([.05,.10,.006,.004,.003]),
    np.array([-.04,.08,.006,.004,.003])
]

def probe_design_objective(x):
    p0=probe_from_angles(x[0],x[1])
    p1=probe_from_angles(x[2],x[3])
    vals=[]
    for th in probe_design_points:
        for kap in FISHER_KAPPA_GRID:
            try:
                F=fisher_for_probe_pair_with_kappa_nuisance(
                    th,p0,p1,kappa_value=kap
                )
                Fe=effective_target_fisher(F)
                sign,logdet=np.linalg.slogdet(Fe)
                if sign<=0 or not np.isfinite(logdet):
                    return 1e6
                vals.append(logdet)
            except Exception:
                return 1e6
    return -float(np.mean(vals))

probe_bounds=[(0,np.pi),(-np.pi,np.pi)]*2
probe_opt_result=differential_evolution(
    probe_design_objective,
    probe_bounds,
    seed=2402,
    popsize=6,
    maxiter=12,
    polish=True,
    workers=1
)

xopt=probe_opt_result.x
optimized_probes=[
    probe_from_angles(xopt[0],xopt[1]),
    probe_from_angles(xopt[2],xopt[3])
]
fixed_probes=[probe0,probe1]

def crlb_summary(probes,label):

    local_rows=[]
    for kap in FISHER_KAPPA_GRID:
        F=fisher_for_probe_pair_with_kappa_nuisance(
            probe_opt_theta,probes[0],probes[1],kappa_value=kap
        )
        Fe=effective_target_fisher(F)
        sign,logdet=np.linalg.slogdet(Fe)
        if sign<=0 or not np.isfinite(logdet):
            raise ValueError(f"Non-positive effective Fisher determinant at kappa={kap}")
        cov=np.linalg.pinv(Fe,rcond=1e-10)
        local_rows.append({
            "kappa":kap,
            "logdet":float(logdet),
            "crlb_var_eps":float(max(cov[0,0],0)),
            "crlb_var_d":float(max(cov[1,1],0))
        })

    local=pd.DataFrame(local_rows)
    return {
        "design":label,
        "kappa_range":"[0.90,1.10] (kappa treated as nuisance)",
        "mean_logdet_effective_Fisher":float(local.logdet.mean()),
        "worst_case_logdet_effective_Fisher":float(local.logdet.min()),
        "mean_CRLB_std_eps":float(np.sqrt(local.crlb_var_eps).mean()),
        "mean_CRLB_std_d":float(np.sqrt(local.crlb_var_d).mean()),
        "worst_case_CRLB_std_eps":float(np.sqrt(local.crlb_var_eps).max()),
        "worst_case_CRLB_std_d":float(np.sqrt(local.crlb_var_d).max()),
        "probe0_alpha":float(np.arccos(
            np.clip(abs(probes[0][0])**2-abs(probes[0][1])**2,-1,1)
        )),
        "probe0_phase":float(
            np.angle(probes[0][1])-np.angle(probes[0][0])
        ),
        "probe1_alpha":float(np.arccos(
            np.clip(abs(probes[1][0])**2-abs(probes[1][1])**2,-1,1)
        )),
        "probe1_phase":float(
            np.angle(probes[1][1])-np.angle(probes[1][0])
        )
    }

probe_design_df=pd.DataFrame([
    crlb_summary(fixed_probes,"fixed_baseline"),
    crlb_summary(optimized_probes,"Fisher_D_optimal")
])
display(probe_design_df)

print("CRLB/probe design uses a 6-parameter Fisher model with kappa as nuisance; the existing 5-parameter identifiability audit keeps kappa fixed.")


In [ ]:

# EXPERIMENT 3 — Reservoir explainability

from sklearn.decomposition import PCA
from scipy.stats import spearmanr
from sklearn.metrics import silhouette_score

# Use a fixed, moderate subset for visualization to keep plots readable.
rng_exp = np.random.default_rng(2403)
n_vis = min(1800, len(dyn_test_X))
vis_idx = rng_exp.choice(len(dyn_test_X), n_vis, replace=False)

Xvis = dyn_test_X[vis_idx]
Yvis = dyn_test_Y[vis_idx]
Hvis = qml_ensemble_features_v24(Xvis, BEST_QML_SEEDS, .02)

pca = PCA(n_components=3, random_state=2403)
H_pca = pca.fit_transform(Hvis)

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(H_pca[:, 0], H_pca[:, 1], c=Yvis[:, 1], s=10, alpha=.65)
ax.set_xlabel("Reservoir PC1")
ax.set_ylabel("Reservoir PC2")
ax.set_title("Reservoir hidden-state geometry colored by EP drift $d_{EP}$")
plt.colorbar(sc, ax=ax, label="$d_{EP}$")
plt.tight_layout()
plt.show()

pca_rows = []
for j in range(3):
    rho, pval = spearmanr(H_pca[:, j], Yvis[:, 1])
    pca_rows.append({
        "component": f"PC{j+1}",
        "explained_variance_ratio": pca.explained_variance_ratio_[j],
        "spearman_rho_with_d_EP": rho,
        "spearman_p": pval
    })
pca_geometry_df = pd.DataFrame(pca_rows)
display(pca_geometry_df)

# Optional UMAP: the notebook remains runnable if umap-learn is not installed.
try:
    import umap
    reducer = umap.UMAP(
        n_components=2, n_neighbors=30, min_dist=.15,
        metric="euclidean", random_state=2403
    )
    H_umap = reducer.fit_transform(Hvis)
    fig, ax = plt.subplots(figsize=(7, 5))
    sc = ax.scatter(H_umap[:, 0], H_umap[:, 1], c=Yvis[:, 1], s=10, alpha=.65)
    ax.set_xlabel("UMAP1")
    ax.set_ylabel("UMAP2")
    ax.set_title("Reservoir hidden-state geometry: UMAP")
    plt.colorbar(sc, ax=ax, label="$d_{EP}$")
    plt.tight_layout()
    plt.show()
    umap_status = "available"
except ImportError:
    H_umap = None
    umap_status = "umap-learn not installed; PCA is the primary geometry analysis."

print("UMAP status:", umap_status)

# Quantify separation across low/mid/high drift bins.
bins = np.quantile(Yvis[:, 1], [0, 1/3, 2/3, 1])
labels = np.digitize(Yvis[:, 1], bins[1:-1])
if len(np.unique(labels)) > 1:
    sil = silhouette_score(H_pca[:, :2], labels)
else:
    sil = np.nan
print("Silhouette score of low/mid/high d_EP bins in PCA space:", sil)

# ---------- Feature importance by grouped permutation ----------
# Fit an independent readout on the dynamic training set.
Htr = qml_ensemble_features_v24(dyn_train_X, BEST_QML_SEEDS, .02)
mu_exp, sd_exp = fit_scaler_v24(Htr)
Htr_s = apply_scaler_v24(Htr, mu_exp, sd_exp)
Hte_s = apply_scaler_v24(Hvis, mu_exp, sd_exp)

exp_model = Ridge(alpha=1e-2).fit(Htr_s, dyn_train_Y[0:len(dyn_train_Y)])

n_imp = min(600, len(dyn_test_X))
imp_idx = rng_exp.choice(len(dyn_test_X), n_imp, replace=False)
Himp = qml_ensemble_features_v24(dyn_test_X[imp_idx], BEST_QML_SEEDS, .02)
Yimp = dyn_test_Y[imp_idx]
Himp_s = apply_scaler_v24(Himp, mu_exp, sd_exp)

pred0 = exp_model.predict(Himp_s)
base_rmse = np.sqrt(np.mean((pred0 - Yimp)**2, axis=0))
base_score = float(np.mean(base_rmse))


NQOBS = len(QOBS)
NT = len(TIMES)

importance_rows = []
for m, seed in enumerate(BEST_QML_SEEDS):
    seed_offset = m * (NT * NQOBS + NQOBS)
    for t in range(NT):
        sl = slice(seed_offset + t*NQOBS, seed_offset + (t+1)*NQOBS)
        Xperm = Himp_s.copy()
        Xperm[:, sl] = Xperm[rng_exp.permutation(n_imp), sl]
        pp = exp_model.predict(Xperm)
        rmse = np.sqrt(np.mean((pp - Yimp)**2, axis=0))
        importance_rows.append({
            "reservoir_seed": seed,
            "time_index": t,
            "time": TIMES[t],
            "delta_RMSE_eps": rmse[0] - base_rmse[0],
            "delta_RMSE_d": rmse[1] - base_rmse[1],
            "delta_mean_RMSE": float(np.mean(rmse) - base_score)
        })

feature_importance_df = pd.DataFrame(importance_rows)
display(
    feature_importance_df
    .sort_values("delta_mean_RMSE", ascending=False)
    .head(15)
)

# Aggregate importance across ensemble members.
fi_summary = (
    feature_importance_df
    .groupby("time")[["delta_RMSE_eps", "delta_RMSE_d", "delta_mean_RMSE"]]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(fi_summary["time"], fi_summary["delta_mean_RMSE"], marker="o")
ax.set_xlabel("Reservoir measurement time")
ax.set_ylabel("Increase in mean RMSE after permutation")
ax.set_title("Reservoir temporal feature importance")
plt.tight_layout()
plt.show()

pca_geometry_df.to_csv("V2_4_reservoir_PCA_geometry.csv", index=False)
feature_importance_df.to_csv("V2_4_reservoir_feature_importance.csv", index=False)


In [ ]:

# EXPERIMENT 4 — Scalability benchmark

from scipy.linalg import expm
from scipy.stats import linregress



def H_scaling(D,theta,kappa=1.0):
    """Controlled D-dimensional non-Hermitian extension for scaling only."""
    eps,d,dk,dw,loss=theta
    k=kappa*(1+dk); gamma=k+d
    H=np.zeros((D,D),dtype=complex)
    H[0,0]=dw+eps+1j*gamma-1j*loss
    H[1,1]=dw-eps-1j*gamma-1j*loss
    H[0,1]=H[1,0]=k
    for j in range(2,D):
        H[j,j]=.015*j-1j*loss
        H[j-1,j]=H[j,j-1]=.35*k
    return H

def scaling_probe_state(D,probe_kind=0):
    psi=np.zeros(D,dtype=complex)
    if probe_kind==0:
        psi[0]=1.0
    else:
        psi[0]=1/np.sqrt(2); psi[1]=1/np.sqrt(2)
    return psi

def reservoir_physics_cost(D,theta,repeats=3):
    H=H_scaling(D,theta)
    times_scale=TIMES[:8]
    t0=time.perf_counter()
    for _ in range(repeats):
        for pk in [0,1]:
            psi0=scaling_probe_state(D,pk)
            for tt in times_scale:
                psi=expm(-1j*H*tt)@psi0
                _=np.abs(psi[:2])**2
                _=np.vdot(psi[:2],psi[:2]).real
    return time.perf_counter()-t0

def spectral_scan_cost(D,theta,scan_points=21,repeats=3):
    t0=time.perf_counter()
    scan=np.linspace(-.08,.08,scan_points)
    for _ in range(repeats):
        for x in scan:
            th=theta.copy(); th[0]=x
            _=np.linalg.eigvals(H_scaling(D,th))
    return time.perf_counter()-t0

scale_theta=np.array([.025,.075,.006,.004,.003])
scale_dims=[2,4,8,12,16,24,32]
scale_rows=[]

for D in scale_dims:
    # Median over repeated timings reduces one-off timing noise.
    res_trials=[reservoir_physics_cost(D,scale_theta,repeats=1) for _ in range(3)]
    spec_trials=[spectral_scan_cost(D,scale_theta,scan_points=21,repeats=1) for _ in range(3)]
    t_res_phys=float(np.median(res_trials))
    t_spec=float(np.median(spec_trials))

    t0=time.perf_counter()
    _=qml_ensemble_features_v24(X_cost[:16],BEST_QML_SEEDS,.02)
    t_qml=time.perf_counter()-t0

    scale_rows.append({
        "system_dimension_D":D,
        "reservoir_physics_time_s":t_res_phys,
        "fixed_QML_readout_time_s_16_samples":t_qml,
        "reservoir_total_time_s":t_res_phys+t_qml,
        "spectral_recalibration_time_s":t_spec,
        "spectral_to_reservoir_ratio":t_spec/max(t_res_phys+t_qml,1e-12),
        "timing_protocol":"median of 3 repeated wall-clock trials"
    })

scalability_df=pd.DataFrame(scale_rows)
display(scalability_df)

valid=scalability_df["system_dimension_D"]>2
res_slope=linregress(
    np.log(scalability_df.loc[valid,"system_dimension_D"]),
    np.log(scalability_df.loc[valid,"reservoir_physics_time_s"])
).slope
spec_slope=linregress(
    np.log(scalability_df.loc[valid,"system_dimension_D"]),
    np.log(scalability_df.loc[valid,"spectral_recalibration_time_s"])
).slope

print(f"Empirical reservoir-physics scaling exponent over tested range: {res_slope:.2f}")
print(f"Empirical spectral-scan scaling exponent over tested range: {spec_slope:.2f}")
print("These exponents are descriptive finite-range measurements, not asymptotic complexity claims.")
print("Both routes use dense linear algebra; dense propagation/eigendecomposition have cubic worst-case scaling in D.")

fig,ax=plt.subplots(figsize=(8,5))
ax.loglog(scalability_df["system_dimension_D"],scalability_df["reservoir_total_time_s"],marker="o",label="Reservoir route")
ax.loglog(scalability_df["system_dimension_D"],scalability_df["spectral_recalibration_time_s"],marker="s",label="Repeated spectral scans")
ax.set_xlabel("System dimension $D$")
ax.set_ylabel("Wall-clock time (s)")
ax.set_title("Computational scaling: reservoir vs spectral recalibration")
ax.legend(); plt.tight_layout(); plt.show()

scalability_df.to_csv("V2_4_scalability_benchmark.csv",index=False)


In [ ]:

# EXPERIMENT 5 — End-to-end validation of Fisher-optimized probes



import time
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge

def make_probe_condition_dataset_v24(n, seed, probes, nphot=NPHOT,
                                     kappa_range=(.90,1.10), paired_thetas=None):
    rng = np.random.default_rng(seed)
    X, Y, K = [], [], []
    if paired_thetas is None:
        paired_thetas = []
        for _ in range(n):
            th, k = sample_theta(rng, True, kappa_range)
            paired_thetas.append((th, k))

    for j, (th, k) in enumerate(paired_thetas):
        # Separate but reproducible noise stream for each sample.
        local_rng = np.random.default_rng(seed + 1000003*j + 17)
        a = noisy_probe(th, probes[0], local_rng, Nphot=nphot, kappa=k)
        b = noisy_probe(th, probes[1], local_rng, Nphot=nphot, kappa=k)
        X.append(np.column_stack([a,b]).ravel())
        Y.append(th[:2])
        K.append(k)
    return np.asarray(X), np.asarray(Y), np.asarray(K), paired_thetas


PAIR_TRAIN_N = 500
PAIR_TEST_N = 250
PAIR_SEED_TRAIN = 120001
PAIR_SEED_TEST = 120002

fixed_probe_pair = [probe0, probe1]
optimized_probe_pair = optimized_probes

# Generate identical latent parameter sets for both probe designs.
_, Ytr_pair, Ktr_pair, train_thetas = make_probe_condition_dataset_v24(
    PAIR_TRAIN_N, PAIR_SEED_TRAIN, fixed_probe_pair, nphot=NPHOT
)
_, Yte_pair, Kte_pair, test_thetas = make_probe_condition_dataset_v24(
    PAIR_TEST_N, PAIR_SEED_TEST, fixed_probe_pair, nphot=NPHOT
)

def train_and_eval_probe_design_v24(probes, label, nphot=NPHOT):
    Xtr, Ytr, _, _ = make_probe_condition_dataset_v24(
        PAIR_TRAIN_N, PAIR_SEED_TRAIN, probes, nphot=nphot,
        paired_thetas=train_thetas
    )
    Xte, Yte, _, _ = make_probe_condition_dataset_v24(
        PAIR_TEST_N, PAIR_SEED_TEST, probes, nphot=nphot,
        paired_thetas=test_thetas
    )

    t0 = time.perf_counter()
    A = qml_ensemble_features_v24(Xtr, BEST_QML_SEEDS, .02)
    T = qml_ensemble_features_v24(Xte, BEST_QML_SEEDS, .02)
    feature_time = time.perf_counter() - t0

    mu, sd = fit_scaler_v24(A)
    A = apply_scaler_v24(A, mu, sd)
    T = apply_scaler_v24(T, mu, sd)

    model = Ridge(alpha=1e-2).fit(A, Ytr)
    pred = model.predict(T)

    out = metrics_v24(Yte, pred)
    out.update({
        "probe_design": label,
        "Nphot_per_population_channel": int(nphot),
        "train_samples": PAIR_TRAIN_N,
        "test_samples": PAIR_TEST_N,
        "feature_time_s": float(feature_time)
    })
    return out

probe_e2e_rows = [
    train_and_eval_probe_design_v24(fixed_probe_pair, "fixed_baseline", NPHOT),
    train_and_eval_probe_design_v24(optimized_probe_pair, "Fisher_D_optimal", NPHOT)
]
probe_e2e_df = pd.DataFrame(probe_e2e_rows)

display(probe_e2e_df[[
    "probe_design","Nphot_per_population_channel",
    "RMSE_eps","RMSE_d","MAE_eps","MAE_d","R2_eps","R2_d",
    "feature_time_s"
]])

# Relative improvement: positive means Fisher-optimized probes are better.
base = probe_e2e_df.loc[probe_e2e_df.probe_design=="fixed_baseline"].iloc[0]
opt  = probe_e2e_df.loc[probe_e2e_df.probe_design=="Fisher_D_optimal"].iloc[0]

e2e_gain = pd.DataFrame([{
    "Nphot": int(NPHOT),
    "RMSE_eps_reduction_pct": 100*(base.RMSE_eps-opt.RMSE_eps)/base.RMSE_eps,
    "RMSE_d_reduction_pct": 100*(base.RMSE_d-opt.RMSE_d)/base.RMSE_d,
    "MAE_eps_reduction_pct": 100*(base.MAE_eps-opt.MAE_eps)/base.MAE_eps,
    "MAE_d_reduction_pct": 100*(base.MAE_d-opt.MAE_d)/base.MAE_d,
    "R2_eps_gain": opt.R2_eps-base.R2_eps,
    "R2_d_gain": opt.R2_d-base.R2_d
}])
display(e2e_gain)

def fit_probe_model_once_v24(probes, nphot=NPHOT):
    """
    Train the probe-specific QML readout once at the reference photon budget.

    The training latent parameter set is exactly the same paired set used
    for the fixed-vs-optimized comparison, so the photon-budget sweep changes
    only the test measurement noise/budget and not the underlying test cases.
    """
    Xtr, Ytr, _, _ = make_probe_condition_dataset_v24(
        PAIR_TRAIN_N,
        PAIR_SEED_TRAIN,
        probes,
        nphot=nphot,
        paired_thetas=train_thetas
    )

    A = qml_ensemble_features_v24(Xtr, BEST_QML_SEEDS, .02)
    mu, sd = fit_scaler_v24(A)
    A_scaled = apply_scaler_v24(A, mu, sd)

    model = Ridge(alpha=1e-2).fit(A_scaled, Ytr)
    return model, mu, sd



# Photon-budget robustness.
# The two models are trained once at the main experimental budget.
# We then reduce/increase the photon budget at test time to assess
# how the two probe designs degrade under measurement noise.

PHOTON_SWEEP = sorted(set([
    max(100, int(NPHOT//10)),
    max(300, int(NPHOT//3)),
    int(NPHOT),
    int(3*NPHOT)
]))

fixed_model, fixed_mu, fixed_sd = fit_probe_model_once_v24(fixed_probe_pair, NPHOT)
opt_model, opt_mu, opt_sd = fit_probe_model_once_v24(optimized_probe_pair, NPHOT)

sweep_rows = []
for nph in PHOTON_SWEEP:
    for probes, label, model, mu, sd in [
        (fixed_probe_pair, "fixed_baseline", fixed_model, fixed_mu, fixed_sd),
        (optimized_probe_pair, "Fisher_D_optimal", opt_model, opt_mu, opt_sd)
    ]:
        Xte, Yte, _, _ = make_probe_condition_dataset_v24(
            PAIR_TEST_N, PAIR_SEED_TEST, probes, nphot=nph,
            paired_thetas=test_thetas
        )
        T = qml_ensemble_features_v24(Xte, BEST_QML_SEEDS, .02)
        T = apply_scaler_v24(T, mu, sd)
        pred = model.predict(T)
        m = metrics_v24(Yte, pred)
        sweep_rows.append({"Nphot": int(nph), "probe_design": label, **m})

probe_budget_df = pd.DataFrame(sweep_rows)
display(probe_budget_df[[
    "Nphot","probe_design","RMSE_eps","RMSE_d","R2_eps","R2_d"
]])

fig, ax = plt.subplots(figsize=(7,4.5))
for label, g in probe_budget_df.groupby("probe_design"):
    ax.plot(g.Nphot, g.RMSE_d, marker="o", label=label)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Photons per population channel")
ax.set_ylabel("RMSE($d_{EP}$)")
ax.set_title("End-to-end sensing accuracy vs measurement budget")
ax.legend()
fig.tight_layout()
plt.show()

probe_e2e_df.to_csv("V2_4_probe_optimization_end_to_end.csv", index=False)
e2e_gain.to_csv("V2_4_probe_optimization_end_to_end_gain.csv", index=False)
probe_budget_df.to_csv("V2_4_probe_optimization_photon_budget.csv", index=False)

print("Completed: fixed vs Fisher-D-optimal probes at identical photon budget.")
print("Important: the optimized probes are validated here separately and do not replace the main V2.4 benchmark.")


In [ ]:


import pandas as pd
import numpy as np

print("=" * 70)
print("FINAL MANUSCRIPT RESULTS EXTRACTION")
print("=" * 70)



# 1. MAIN BENCHMARK

print("\n[1] MAIN BENCHMARK")
print("-" * 70)

names = [
    "benchmark_df",
    "results_df",
    "main_results_df",
    "summary_df"
]

for name in names:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, (pd.DataFrame, pd.Series)):
            print(f"\n--- {name} ---")
            display(obj)



# 2. FISHER / CRLB

print("\n[2] FISHER OPTIMIZATION")
print("-" * 70)

names = [
    "fisher_df",
    "fisher_results",
    "fisher_summary",
    "crlb_df",
    "crlb_results",
    "fisher_opt_df",
    "probe_opt_df",
    "optimized_probe_df"
]

for name in names:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, (pd.DataFrame, pd.Series)):
            print(f"\n--- {name} ---")
            display(obj)



# 3. END-TO-END FISHER PROBE VALIDATION

print("\n[3] END-TO-END FISHER PROBE VALIDATION")
print("-" * 70)

names = [
    "probe_e2e_df",
    "e2e_gain",
    "probe_budget_df"
]

for name in names:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, (pd.DataFrame, pd.Series)):
            print(f"\n--- {name} ---")
            display(obj)



# 4. MEASUREMENT COST

print("\n[4] MEASUREMENT COST")
print("-" * 70)

names = [
    "measurement_cost_df",
    "cost_df",
    "measurement_df",
    "acquisition_df",
    "cost_summary"
]

for name in names:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, (pd.DataFrame, pd.Series)):
            print(f"\n--- {name} ---")
            display(obj)



# 5. OOD / UNSEEN DEVICE

print("\n[5] UNSEEN-DEVICE / OOD PERFORMANCE")
print("-" * 70)

names = [
    "strict_ood",
    "strict_ood_df",
    "ood_df",
    "ood_results",
    "unseen_device_df",
    "unseen_device_results"
]

for name in names:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, (pd.DataFrame, pd.Series)):
            print(f"\n--- {name} ---")
            display(obj)



# 6. INDEPENDENT DYNAMIC TRACKING

print("\n[6] INDEPENDENT DYNAMIC TRACKING")
print("-" * 70)

names = [
    "dynamic_results",
    "dynamic_df",
    "dynamic_tracking_df",
    "independent_dynamic_df",
    "independent_dynamic_results"
]

for name in names:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, (pd.DataFrame, pd.Series)):
            print(f"\n--- {name} ---")
            display(obj)



# 7. CONTROLLER VALIDATION

print("\n[7] INDEPENDENT CONTROLLER VALIDATION")
print("-" * 70)

names = [
    "strict_controller",
    "strict_controller_df",
    "controller_results",
    "controller_df",
    "restoration_df",
    "restoration_results"
]

for name in names:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, (pd.DataFrame, pd.Series)):
            print(f"\n--- {name} ---")
            display(obj)



# 8. BIAS / BOOTSTRAP

print("\n[8] BIAS / BOOTSTRAP RESULTS")
print("-" * 70)

names = [
    "bias_df",
    "bias_results",
    "bootstrap_df",
    "bootstrap_results",
    "bias_bootstrap_df"
]

for name in names:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, (pd.DataFrame, pd.Series)):
            print(f"\n--- {name} ---")
            display(obj)



# 9. FEATURE IMPORTANCE

print("\n[9] FEATURE IMPORTANCE")
print("-" * 70)

names = [
    "feature_importance_df",
    "feature_importance",
    "importance_df",
    "temporal_importance_df"
]

for name in names:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, (pd.DataFrame, pd.Series)):
            print(f"\n--- {name} ---")
            display(obj)



# 10. SCALABILITY

print("\n[10] SCALABILITY")
print("-" * 70)

names = [
    "scaling_df",
    "scalability_df",
    "scaling_results",
    "complexity_df",
    "scaling_summary"
]

for name in names:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, (pd.DataFrame, pd.Series)):
            print(f"\n--- {name} ---")
            display(obj)


print("\n" + "=" * 70)
print("EXTRACTION COMPLETE")
print("=" * 70)

In [ ]:

# PLOT SET 1 — Main benchmark and strict OOD comparison

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


metric_specs = [
    ("RMSE_eps", r"RMSE($\epsilon$)"),
    ("RMSE_d", r"RMSE($d_{EP}$)"),
    ("MAE_eps", r"MAE($\epsilon$)"),
    ("MAE_d", r"MAE($d_{EP}$)"),
    ("R2_eps", r"$R^2(\epsilon)$"),
    ("R2_d", r"$R^2(d_{EP})$")
]

bench_summary = (
    strict_benchmark
    .groupby("model")[["RMSE_eps","RMSE_d","MAE_eps","MAE_d","R2_eps","R2_d"]]
    .agg(["mean","std"])
)

model_order = ["QML_reservoir", "classical_reservoir", "ridge"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, metric_pair, title in zip(
    axes,
    [("RMSE_eps","RMSE_d"), ("R2_eps","R2_d")],
    ["Main benchmark: RMSE", "Main benchmark: $R^2$"]
):
    x = np.arange(2)
    width = 0.24
    for j, model in enumerate(model_order):
        vals = [
            bench_summary.loc[model, (metric_pair[0], "mean")],
            bench_summary.loc[model, (metric_pair[1], "mean")]
        ]
        errs = [
            bench_summary.loc[model, (metric_pair[0], "std")],
            bench_summary.loc[model, (metric_pair[1], "std")]
        ]
        ax.bar(x + (j-1)*width, vals, width,
               yerr=errs, capsize=4, label=model.replace("_", " "))
    ax.set_xticks(x)
    ax.set_xticklabels([r"$\epsilon$", r"$d_{EP}$"])
    ax.set_title(title)
    ax.grid(axis="y", alpha=.25)

axes[0].set_ylabel("Error")
axes[1].set_ylabel(r"$R^2$")
axes[1].legend(frameon=False)
fig.tight_layout()
plt.show()



ood_summary = (
    strict_ood
    .groupby("model")[["RMSE_eps","RMSE_d","R2_eps","R2_d"]]
    .agg(["mean","std"])
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, metrics, title in zip(
    axes,
    [("RMSE_eps","RMSE_d"), ("R2_eps","R2_d")],
    ["Unseen-device test: RMSE", "Unseen-device test: $R^2$"]
):
    x = np.arange(2)
    width = 0.24
    for j, model in enumerate(model_order):
        vals = [ood_summary.loc[model, (m, "mean")] for m in metrics]
        errs = [ood_summary.loc[model, (m, "std")] for m in metrics]
        ax.bar(x + (j-1)*width, vals, width,
               yerr=errs, capsize=4, label=model.replace("_", " "))
    ax.set_xticks(x)
    ax.set_xticklabels([r"$\epsilon$", r"$d_{EP}$"])
    ax.set_title(title)
    ax.grid(axis="y", alpha=.25)

axes[0].set_ylabel("Error")
axes[1].set_ylabel(r"$R^2$")
axes[1].legend(frameon=False)
fig.tight_layout()
plt.show()


In [ ]:

# PLOT SET 2 — Independent dynamic tracking



T_plot = np.arange(len(dyn_test_Y))
# Use one complete held-out trajectory for visualization.
# The strict test contains 30 trajectories × 120 points.
steps_per_traj = 120
traj_id = 0
sl = slice(traj_id * steps_per_traj, (traj_id + 1) * steps_per_traj)

fig, axes = plt.subplots(2, 1, figsize=(9, 6.5), sharex=True)

axes[0].plot(T_plot[sl], dyn_test_Y[sl, 1], label=r"True $d_{EP}$")
axes[0].plot(T_plot[sl], dyn_pred[sl, 1], "--", label=r"Estimated $d_{EP}$")
axes[0].set_ylabel(r"$d_{EP}$")
axes[0].set_title("Independent dynamic tracking on an unseen trajectory")
axes[0].legend(frameon=False)
axes[0].grid(alpha=.25)

axes[1].plot(T_plot[sl], dyn_test_Y[sl, 0], label=r"True $\epsilon$")
axes[1].plot(T_plot[sl], dyn_pred[sl, 0], "--", label=r"Estimated $\epsilon$")
axes[1].set_xlabel("Test trajectory sample")
axes[1].set_ylabel(r"$\epsilon$")
axes[1].legend(frameon=False)
axes[1].grid(alpha=.25)

fig.tight_layout()
plt.show()

# Error distributions for the complete independent dynamic test.
err_eps = dyn_pred[:,0] - dyn_test_Y[:,0]
err_d = dyn_pred[:,1] - dyn_test_Y[:,1]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(err_eps, bins=35)
axes[0].axvline(0, linestyle="--")
axes[0].set_xlabel(r"$\hat{\epsilon}-\epsilon$")
axes[0].set_ylabel("Count")
axes[0].set_title(r"Dynamic tracking error: $\epsilon$")

axes[1].hist(err_d, bins=35)
axes[1].axvline(0, linestyle="--")
axes[1].set_xlabel(r"$\hat d_{EP}-d_{EP}$")
axes[1].set_ylabel("Count")
axes[1].set_title(r"Dynamic tracking error: $d_{EP}$")

fig.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# PLOT SET 3 — Fisher information / CRLB result representation

fd = probe_design_df.set_index("design")

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0,0].bar(["Fixed", "Fisher-D"], [
    fd.loc["fixed_baseline", "mean_logdet_effective_Fisher"],
    fd.loc["Fisher_D_optimal", "mean_logdet_effective_Fisher"]
])
axes[0,0].set_ylabel(r"$\log\det F_{\mathrm{eff}}$")
axes[0,0].set_title("Effective Fisher information")

axes[0,1].bar(["Fixed", "Fisher-D"], [
    np.exp(fd.loc["fixed_baseline", "mean_logdet_effective_Fisher"]),
    np.exp(fd.loc["Fisher_D_optimal", "mean_logdet_effective_Fisher"])
])
axes[0,1].set_yscale("log")
axes[0,1].set_ylabel(r"$\det F_{\mathrm{eff}}$")
axes[0,1].set_title("Effective Fisher determinant")

axes[1,0].bar(["Fixed", "Fisher-D"], [
    fd.loc["fixed_baseline", "mean_CRLB_std_eps"],
    fd.loc["Fisher_D_optimal", "mean_CRLB_std_eps"]
])
axes[1,0].set_ylabel(r"CRLB std. of $\epsilon$")
axes[1,0].set_title(r"CRLB: $\epsilon$")

axes[1,1].bar(["Fixed", "Fisher-D"], [
    fd.loc["fixed_baseline", "mean_CRLB_std_d"],
    fd.loc["Fisher_D_optimal", "mean_CRLB_std_d"]
])
axes[1,1].set_ylabel(r"CRLB std. of $d_{EP}$")
axes[1,1].set_title(r"CRLB: $d_{EP}$")

for ax in axes.ravel():
    ax.grid(axis="y", alpha=.25)

fig.tight_layout()
plt.show()


# End-to-end Fisher validation: all six reported metrics.
e2e = probe_e2e_df.set_index("probe_design")
metric_names = [
    ("RMSE_eps", r"RMSE($\epsilon$)"),
    ("RMSE_d", r"RMSE($d_{EP}$)"),
    ("MAE_eps", r"MAE($\epsilon$)"),
    ("MAE_d", r"MAE($d_{EP}$)"),
    ("R2_eps", r"$R^2(\epsilon)$"),
    ("R2_d", r"$R^2(d_{EP})$")
]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, (metric, label) in zip(axes.ravel(), metric_names):
    vals = [
        e2e.loc["fixed_baseline", metric],
        e2e.loc["Fisher_D_optimal", metric]
    ]
    ax.bar(["Fixed", "Fisher-D"], vals)
    ax.set_title(label)
    ax.grid(axis="y", alpha=.25)

fig.suptitle("End-to-end fixed vs Fisher-D-optimal probe validation", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:

# PLOT SET 4 — Photon-budget robustness

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

plot_specs = [
    ("RMSE_eps", r"RMSE($\epsilon$)"),
    ("RMSE_d", r"RMSE($d_{EP}$)"),
    ("R2_eps", r"$R^2(\epsilon)$"),
    ("R2_d", r"$R^2(d_{EP})$")
]

for ax, (metric, ylabel) in zip(axes.ravel(), plot_specs):
    for label, g in probe_budget_df.groupby("probe_design"):
        g = g.sort_values("Nphot")
        ax.plot(g["Nphot"], g[metric], marker="o",
                label=label.replace("_", " "))
    ax.set_xscale("log")
    if metric.startswith("RMSE"):
        ax.set_yscale("log")
    ax.set_xlabel("Photons per population channel")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=.25)

axes[0,0].legend(frameon=False)
fig.suptitle("End-to-end robustness to measurement budget", y=1.02)
fig.tight_layout()
plt.show()


# Relative RMSE reduction from Fisher-D-optimal probes.
rows = []
for nph in sorted(probe_budget_df["Nphot"].unique()):
    g = probe_budget_df[probe_budget_df["Nphot"] == nph].set_index("probe_design")
    rows.append({
        "Nphot": nph,
        "RMSE_eps_reduction_pct":
            100*(g.loc["fixed_baseline","RMSE_eps"] -
                  g.loc["Fisher_D_optimal","RMSE_eps"]) /
            g.loc["fixed_baseline","RMSE_eps"],
        "RMSE_d_reduction_pct":
            100*(g.loc["fixed_baseline","RMSE_d"] -
                  g.loc["Fisher_D_optimal","RMSE_d"]) /
            g.loc["fixed_baseline","RMSE_d"]
    })

photon_gain_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(photon_gain_df.Nphot, photon_gain_df.RMSE_eps_reduction_pct,
        marker="o", label=r"$\epsilon$")
ax.plot(photon_gain_df.Nphot, photon_gain_df.RMSE_d_reduction_pct,
        marker="s", label=r"$d_{EP}$")
ax.axhline(0, linestyle="--")
ax.set_xscale("log")
ax.set_xlabel("Photons per population channel")
ax.set_ylabel("RMSE reduction (%)")
ax.set_title("Relative benefit of Fisher-D-optimal probes")
ax.legend(frameon=False)
ax.grid(alpha=.25)
fig.tight_layout()
plt.show()

photon_gain_df.to_csv("V2_4_photon_budget_relative_gain.csv", index=False)


In [ ]:

# PLOT SET 5 — Reservoir geometry and temporal importance

# PCA explained variance
fig, ax = plt.subplots(figsize=(7, 4.5))
components = np.arange(1, len(pca.explained_variance_ratio_) + 1)
ax.bar(components, pca.explained_variance_ratio_ * 100)
ax.set_xlabel("Principal component")
ax.set_ylabel("Explained variance (%)")
ax.set_title("Explained variance of reservoir hidden-state representation")
ax.set_xticks(components)
ax.grid(axis="y", alpha=.25)
fig.tight_layout()
plt.show()


# PC1 vs EP drift with rank correlation
rho, pval = spearmanr(H_pca[:,0], Yvis[:,1])

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(H_pca[:,0], Yvis[:,1], s=12, alpha=.55)
coef = np.polyfit(H_pca[:,0], Yvis[:,1], 1)
xx = np.linspace(H_pca[:,0].min(), H_pca[:,0].max(), 200)
ax.plot(xx, coef[0]*xx + coef[1], "--")
ax.set_xlabel("Reservoir PC1")
ax.set_ylabel(r"$d_{EP}$")
ax.set_title(fr"PC1 vs EP drift (Spearman $\rho$={rho:.4f}, p={pval:.2e})")
ax.grid(alpha=.25)
fig.tight_layout()
plt.show()


# Separate importance for epsilon and d_EP
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(fi_summary["time"], fi_summary["delta_RMSE_eps"], marker="o")
axes[0].axhline(0, linestyle="--")
axes[0].set_xlabel("Reservoir measurement time")
axes[0].set_ylabel(r"$\Delta$ RMSE after permutation")
axes[0].set_title(r"Temporal importance for $\epsilon$")
axes[0].grid(alpha=.25)

axes[1].plot(fi_summary["time"], fi_summary["delta_RMSE_d"], marker="o")
axes[1].axhline(0, linestyle="--")
axes[1].set_xlabel("Reservoir measurement time")
axes[1].set_ylabel(r"$\Delta$ RMSE after permutation")
axes[1].set_title(r"Temporal importance for $d_{EP}$")
axes[1].grid(alpha=.25)

fig.tight_layout()
plt.show()


In [ ]:

#   Detector/phase-noise robustness



NOISE_TEST_N = 250
NOISE_SEED = 130001
DETECTOR_NOISE_SWEEP = [0.0, 0.001, 0.002, 0.005, 0.01, 0.02]
PHASE_NOISE_SWEEP = [0.0, 0.005, 0.01, 0.02, 0.05, 0.10]

# Reuse the same latent test cases for every noise level.
rng_noise = np.random.default_rng(NOISE_SEED)
noise_thetas = [sample_theta(rng_noise, True, (.90, 1.10))
                for _ in range(NOISE_TEST_N)]

def make_noise_test_data(probes, noise_thetas, sigma_det, sigma_phase):
    X, Y = [], []
    for j, (th, k) in enumerate(noise_thetas):
        local_rng = np.random.default_rng(NOISE_SEED + 900000 + j)
        a = noisy_probe(
            th, probes[0], local_rng, Nphot=NPHOT,
            sigma_det=sigma_det, sigma_phase=sigma_phase, kappa=k
        )
        b = noisy_probe(
            th, probes[1], local_rng, Nphot=NPHOT,
            sigma_det=sigma_det, sigma_phase=sigma_phase, kappa=k
        )
        X.append(np.column_stack([a,b]).ravel())
        Y.append(th[:2])
    return np.asarray(X), np.asarray(Y)

# Models are trained once at the reference noise setting.
fixed_noise_model, fixed_noise_mu, fixed_noise_sd = \
    fit_probe_model_once_v24(fixed_probe_pair, NPHOT)

opt_noise_model, opt_noise_mu, opt_noise_sd = \
    fit_probe_model_once_v24(optimized_probe_pair, NPHOT)

noise_rows = []

for noise_type, values in [
    ("detector", DETECTOR_NOISE_SWEEP),
    ("phase", PHASE_NOISE_SWEEP)
]:
    for value in values:
        sigma_det = value if noise_type == "detector" else SIGMA_DET
        sigma_phase = value if noise_type == "phase" else SIGMA_PHASE

        for probes, label, model, mu, sd in [
            (fixed_probe_pair, "fixed_baseline",
             fixed_noise_model, fixed_noise_mu, fixed_noise_sd),
            (optimized_probe_pair, "Fisher_D_optimal",
             opt_noise_model, opt_noise_mu, opt_noise_sd)
        ]:
            Xn, Yn = make_noise_test_data(
                probes, noise_thetas, sigma_det, sigma_phase
            )
            Hn = qml_ensemble_features_v24(Xn, BEST_QML_SEEDS, .02)
            Hn = apply_scaler_v24(Hn, mu, sd)
            pred = model.predict(Hn)
            mm = metrics_v24(Yn, pred)

            noise_rows.append({
                "noise_type": noise_type,
                "noise_level": value,
                "probe_design": label,
                **mm
            })

noise_robustness_df = pd.DataFrame(noise_rows)
display(noise_robustness_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for label, g in noise_robustness_df[
    noise_robustness_df.noise_type == "detector"
].groupby("probe_design"):
    g = g.sort_values("noise_level")
    axes[0].plot(g.noise_level, g.RMSE_d, marker="o",
                 label=label.replace("_", " "))

for label, g in noise_robustness_df[
    noise_robustness_df.noise_type == "phase"
].groupby("probe_design"):
    g = g.sort_values("noise_level")
    axes[1].plot(g.noise_level, g.RMSE_d, marker="o",
                 label=label.replace("_", " "))

axes[0].set_xlabel(r"Detector noise $\sigma_{\rm det}$")
axes[0].set_ylabel(r"RMSE($d_{EP}$)")
axes[0].set_title(r"Detector-noise robustness")
axes[0].grid(alpha=.25)

axes[1].set_xlabel(r"Phase noise $\sigma_{\rm phase}$")
axes[1].set_ylabel(r"RMSE($d_{EP}$)")
axes[1].set_title(r"Phase-noise robustness")
axes[1].grid(alpha=.25)
axes[1].legend(frameon=False)

fig.tight_layout()
plt.show()

noise_robustness_df.to_csv(
    "V2_4_detector_phase_noise_robustness.csv", index=False
)


In [ ]:

# CONTROLLED NOISE ROBUSTNESS

# Purpose:
#   The previous noise experiment intentionally trained at the reference
#   noise level and tested at altered noise. That is a useful distribution-
#   shift test, but it should not be confused with noise-matched robustness.
#
#   Here, for every noise level, each probe design is TRAINED and TESTED
#   at the same noise level. This isolates the effect of measurement noise
#   on the achievable sensing performance.

MATCHED_DETECTOR_SWEEP = [0.0, 0.001, 0.002, 0.005, 0.01, 0.02]
MATCHED_PHASE_SWEEP = [0.0, 0.005, 0.01, 0.02, 0.05, 0.10]

# Use fixed latent parameter sets for paired comparisons.
rng_matched = np.random.default_rng(140001)
matched_train_thetas = [sample_theta(rng_matched, True, (.90, 1.10))
                        for _ in range(PAIR_TRAIN_N)]
matched_test_thetas = [sample_theta(rng_matched, True, (.90, 1.10))
                       for _ in range(PAIR_TEST_N)]

def make_probe_condition_dataset_noise_v24(n, seed, probes, nphot,
                                           sigma_det, sigma_phase,
                                           paired_thetas):
    X, Y, K = [], [], []
    for j, (th, k) in enumerate(paired_thetas):
        local_rng = np.random.default_rng(seed + 2000003*j + 31)
        a = noisy_probe(th, probes[0], local_rng, Nphot=nphot,
                        sigma_det=sigma_det, sigma_phase=sigma_phase,
                        kappa=k)
        b = noisy_probe(th, probes[1], local_rng, Nphot=nphot,
                        sigma_det=sigma_det, sigma_phase=sigma_phase,
                        kappa=k)
        X.append(np.column_stack([a,b]).ravel())
        Y.append(th[:2]); K.append(k)
    return np.asarray(X), np.asarray(Y), np.asarray(K)

def train_eval_at_noise_v24(probes, label, sigma_det, sigma_phase):
    Xtr, Ytr, _ = make_probe_condition_dataset_noise_v24(
        PAIR_TRAIN_N, 140101, probes, NPHOT,
        sigma_det, sigma_phase, matched_train_thetas
    )
    Xte, Yte, _ = make_probe_condition_dataset_noise_v24(
        PAIR_TEST_N, 140201, probes, NPHOT,
        sigma_det, sigma_phase, matched_test_thetas
    )

    Htr = qml_ensemble_features_v24(Xtr, BEST_QML_SEEDS, .02)
    Hte = qml_ensemble_features_v24(Xte, BEST_QML_SEEDS, .02)
    mu, sd = fit_scaler_v24(Htr)
    Htr = apply_scaler_v24(Htr, mu, sd)
    Hte = apply_scaler_v24(Hte, mu, sd)
    model = Ridge(alpha=1e-2).fit(Htr, Ytr)
    pred = model.predict(Hte)
    out = metrics_v24(Yte, pred)
    out.update({
        "probe_design": label,
        "sigma_det": sigma_det,
        "sigma_phase": sigma_phase,
        "training_mode": "noise_matched"
    })
    return out

# Explicitly label each sweep. This prevents the shared baseline
# (sigma_det=0.002, sigma_phase=0.01) from being counted in both sweeps.
matched_rows = []
for value in MATCHED_DETECTOR_SWEEP:
    for probes, label in [
        (fixed_probe_pair, "fixed_baseline"),
        (optimized_probe_pair, "Fisher_D_optimal")
    ]:
        row = train_eval_at_noise_v24(probes, label, value, SIGMA_PHASE)
        row.update({"noise_type": "detector", "noise_level": value})
        matched_rows.append(row)

for value in MATCHED_PHASE_SWEEP:
    for probes, label in [
        (fixed_probe_pair, "fixed_baseline"),
        (optimized_probe_pair, "Fisher_D_optimal")
    ]:
        row = train_eval_at_noise_v24(probes, label, SIGMA_DET, value)
        row.update({"noise_type": "phase", "noise_level": value})
        matched_rows.append(row)

noise_matched_df = pd.DataFrame(matched_rows)
display(noise_matched_df)

#  Matched-training robustness figure
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for label, g in noise_matched_df.groupby("probe_design"):
    gd = g[g.noise_type.eq("detector")].sort_values("noise_level")
    gp = g[g.noise_type.eq("phase")].sort_values("noise_level")
    axes[0].plot(gd.noise_level, gd.RMSE_d, marker="o",
                 label=label.replace("_", " "))
    axes[1].plot(gp.noise_level, gp.RMSE_d, marker="o",
                 label=label.replace("_", " "))

axes[0].set_xlabel(r"Detector noise $\sigma_{\rm det}$")
axes[0].set_ylabel(r"RMSE($d_{EP}$)")
axes[0].set_title("Noise-matched detector robustness")
axes[0].grid(alpha=.25)

axes[1].set_xlabel(r"Phase noise $\sigma_{\rm phase}$")
axes[1].set_ylabel(r"RMSE($d_{EP}$)")
axes[1].set_title("Noise-matched phase robustness")
axes[1].grid(alpha=.25)
axes[1].legend(frameon=False)

fig.tight_layout()
plt.show()



noise_matched_df.to_csv("V2_4_noise_matched_robustness.csv", index=False)


In [ ]:

#   Corrected noise-gain calculation



fixed = noise_matched_df[
    noise_matched_df["probe_design"].eq("fixed_baseline")
].copy()
fisher = noise_matched_df[
    noise_matched_df["probe_design"].eq("Fisher_D_optimal")
].copy()

noise_matched_gain_df = fixed.merge(
    fisher,
    on=["noise_type", "noise_level"],
    suffixes=("_fixed", "_fisher"),
    validate="one_to_one"
)

noise_matched_gain_df["gain_eps_pct"] = 100 * (
    noise_matched_gain_df["RMSE_eps_fixed"] -
    noise_matched_gain_df["RMSE_eps_fisher"]
) / noise_matched_gain_df["RMSE_eps_fixed"]
noise_matched_gain_df["gain_d_pct"] = 100 * (
    noise_matched_gain_df["RMSE_d_fixed"] -
    noise_matched_gain_df["RMSE_d_fisher"]
) / noise_matched_gain_df["RMSE_d_fixed"]

display(noise_matched_gain_df[[
    "noise_type", "noise_level",
    "RMSE_eps_fixed", "RMSE_eps_fisher", "gain_eps_pct",
    "RMSE_d_fixed", "RMSE_d_fisher", "gain_d_pct"
]].sort_values(["noise_type", "noise_level"]))

fig, axes = plt.subplots(2, 2, figsize=(10.2, 7.4), constrained_layout=True)
panels = [
    ("detector", "RMSE_eps", r"$\sigma_{\mathrm{det}}$", r"RMSE($\epsilon$)", r"Detector noise: $\epsilon$"),
    ("detector", "RMSE_d", r"$\sigma_{\mathrm{det}}$", r"RMSE($d_{EP}$)", r"Detector noise: $d_{EP}$"),
    ("phase", "RMSE_eps", r"$\sigma_{\mathrm{phase}}$", r"RMSE($\epsilon$)", r"Phase noise: $\epsilon$"),
    ("phase", "RMSE_d", r"$\sigma_{\mathrm{phase}}$", r"RMSE($d_{EP}$)", r"Phase noise: $d_{EP}$")
]
for ax, (ntype, metric, xlabel, ylabel, title) in zip(axes.flat, panels):
    g = noise_matched_gain_df[
        noise_matched_gain_df["noise_type"].eq(ntype)
    ].sort_values("noise_level")
    x = g["noise_level"].to_numpy()
    yf = g[f"{metric}_fixed"].to_numpy()
    yo = g[f"{metric}_fisher"].to_numpy()
    gains = g["gain_eps_pct" if metric == "RMSE_eps" else "gain_d_pct"].to_numpy()
    ax.plot(x, yf, marker="o", linewidth=1.8, markersize=5, label="Fixed probes")
    ax.plot(x, yo, marker="s", linewidth=1.8, markersize=5, label="Fisher-D-optimal")
    for xi, yi, gi in zip(x, yo, gains):
        ax.annotate(f"{gi:+.0f}%", (xi, yi), xytext=(0, 5),
                    textcoords="offset points", ha="center", fontsize=8)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=.25)
    ax.set_axisbelow(True)
    ax.set_xscale("symlog", linthresh=0.001 if ntype == "detector" else 0.005)
    ax.set_ylim(bottom=0, top=max(yf.max(), yo.max())*1.24)
axes[0,0].legend(frameon=False, loc="upper left")
fig.suptitle("Figure 6. Robustness to detector and phase measurement noise", fontsize=13)
fig.savefig("Figure_6_noise_robustness_publication.png", dpi=600, bbox_inches="tight")
plt.show()


In [ ]:


#   Raw-count Poisson + Gaussian reference Fisher



RAW_FISHER_NOISE_CASES = [
    (0.002, 0.01),
    (0.010, 0.05),
]

def raw_count_reference_fisher(theta, psi0, kappa_value=KAPPA,
                               nphot=NPHOT, sigma_det=SIGMA_DET,
                               sigma_phase=SIGMA_PHASE):
    theta = np.asarray(theta, dtype=float)
    # Parameter vector [epsilon, d_EP, delta_kappa, delta_omega, loss, kappa].
    x0 = np.concatenate([theta, [float(kappa_value)]])

    def raw_means(x):
        th = np.asarray(x[:5], dtype=float)
        kap = float(x[5])
        o = observables(th, TIMES, psi0, kap)
        p1 = np.maximum(o[:,1], 1e-15)
        p2 = np.maximum(o[:,2], 1e-15)
        return np.concatenate([nphot*p1, nphot*p2])

    def gaussian_means(x):
        th = np.asarray(x[:5], dtype=float)
        kap = float(x[5])
        o = observables(th, TIMES, psi0, kap)
        return np.column_stack([o[:,4], o[:,5], o[:,6]]).ravel()

    # Exact Fisher contribution for independent Poisson counts:
    lam = raw_means(x0)
    J_lam = numerical_jacobian(raw_means, x0)
    F_poisson = J_lam.T @ ((1.0/np.maximum(lam,1e-15))[:,None] * J_lam)

    # Gaussian channels with fixed variances.
    J_g = numerical_jacobian(gaussian_means, x0)
    # [cx, cy] have sigma_det, phi has sigma_phase.
    var_g = np.tile([sigma_det**2, sigma_det**2, sigma_phase**2], len(TIMES))
    F_gauss = J_g.T @ ((1.0/np.maximum(var_g,1e-18))[:,None] * J_g)
    return F_poisson + F_gauss


def raw_reference_summary(probes, label, sigma_det=SIGMA_DET,
                          sigma_phase=SIGMA_PHASE):
    rows=[]
    for kap in FISHER_KAPPA_GRID:
        F = (
            raw_count_reference_fisher(
                probe_opt_theta, probes[0], kap,
                sigma_det=sigma_det, sigma_phase=sigma_phase
            ) +
            raw_count_reference_fisher(
                probe_opt_theta, probes[1], kap,
                sigma_det=sigma_det, sigma_phase=sigma_phase
            )
        )
        Fe = effective_target_fisher(F)
        sign, logdet = np.linalg.slogdet(Fe)
        if sign <= 0 or not np.isfinite(logdet):
            raise ValueError(f"Invalid raw-count Fisher at kappa={kap}")
        cov = np.linalg.pinv(Fe, rcond=1e-10)
        rows.append({
            'design': label,
            'kappa': kap,
            'logdet': float(logdet),
            'crlb_std_eps': float(np.sqrt(max(cov[0,0],0))),
            'crlb_std_d': float(np.sqrt(max(cov[1,1],0)),),
        })
    return pd.DataFrame(rows)

raw_ref_tables=[]
for sd,sp in RAW_FISHER_NOISE_CASES:
    for probes,label in [
        (fixed_probe_pair,'fixed_baseline'),
        (optimized_probe_pair,'Fisher_D_optimal')
    ]:
        g=raw_reference_summary(probes,label,sd,sp)
        g['sigma_det']=sd; g['sigma_phase']=sp
        raw_ref_tables.append(g)

raw_poisson_fisher_df=pd.concat(raw_ref_tables,ignore_index=True)
raw_poisson_fisher_summary=(
    raw_poisson_fisher_df.groupby(['sigma_det','sigma_phase','design'])
    .agg(mean_logdet=('logdet','mean'),
         worst_logdet=('logdet','min'),
         mean_crlb_std_eps=('crlb_std_eps','mean'),
         mean_crlb_std_d=('crlb_std_d','mean'))
    .reset_index()
)
display(raw_poisson_fisher_summary)
raw_poisson_fisher_summary.to_csv('V2_6_raw_count_poisson_fisher_reference.csv',index=False)


In [ ]:

#   Noise-aware Fisher probe design
#



NOISE_AWARE_CASES = [
    ('detector', 0.010, SIGMA_PHASE),
    ('phase',    SIGMA_DET, 0.050),
]

# Coarse grid for optimization only
FISHER_KAPPA_GRID_OPT = np.linspace(0.90, 1.10, 3)

# Full grid retained for final Fisher validation
FISHER_KAPPA_GRID_FULL = FISHER_KAPPA_GRID.copy()

NOISE_AWARE_N_RESTARTS = 1
NOISE_AWARE_MAXITER = 12
NOISE_AWARE_POPSIZE = 6
NOISE_AWARE_EVAL_N = min(PAIR_TEST_N, 300)

noise_aware_probe_records = []
noise_aware_probe_pairs = {}
noise_aware_restart_records = []


def make_noise_aware_probe_objective(sigma_det, sigma_phase):
    """
    Objective for noise-aware probe design.

    Uses a reduced kappa grid for optimization to limit runtime.
    Final validation uses the full Fisher_KAPPA_GRID.
    """
    def objective(x):

        p0 = probe_from_angles(x[0], x[1])
        p1 = probe_from_angles(x[2], x[3])

        vals = []

        for th in probe_design_points:

            for kap in FISHER_KAPPA_GRID_OPT:

                try:
                    F = fisher_for_probe_pair_with_kappa_nuisance(
                        th,
                        p0,
                        p1,
                        kappa_value=kap,
                        sigma_det=sigma_det,
                        sigma_phase=sigma_phase
                    )

                    Fe = effective_target_fisher(F)

                    sign, logdet = np.linalg.slogdet(Fe)

                    if sign <= 0 or not np.isfinite(logdet):
                        return 1e6

                    vals.append(logdet)

                except Exception:
                    return 1e6

        if len(vals) == 0 or not np.all(np.isfinite(vals)):
            return 1e6

        return -float(np.mean(vals))

    return objective



# Optimize probes for the selected representative noise cases

for case_id, (ntype, sd, sp) in enumerate(NOISE_AWARE_CASES):

    case_results = []

    objective_fn = make_noise_aware_probe_objective(sd, sp)

    for restart in range(NOISE_AWARE_N_RESTARTS):

        result = differential_evolution(
            objective_fn,
            probe_bounds,
            seed=26000 + 100 * case_id + restart,
            popsize=NOISE_AWARE_POPSIZE,
            maxiter=NOISE_AWARE_MAXITER,
            polish=True,
            workers=1,
            updating='immediate'
        )

        case_results.append(result)

        noise_aware_restart_records.append({
            'noise_type': ntype,
            'sigma_det': sd,
            'sigma_phase': sp,
            'restart': restart,
            'objective_logdet': float(-result.fun),
            'opt_success': bool(result.success),
            'nfev': int(result.nfev),
            'message': str(result.message)
        })


    converged = [
        r for r in case_results
        if r.success and np.isfinite(r.fun)
    ]

    if converged:
        result = min(converged, key=lambda r: r.fun)
        selection_status = "converged"
    else:
        valid = [
            r for r in case_results
            if np.isfinite(r.fun)
        ]

        if not valid:
            raise RuntimeError(
                f"Noise-aware optimization failed for {ntype}, "
                f"sigma_det={sd}, sigma_phase={sp}"
            )

        result = min(valid, key=lambda r: r.fun)
        selection_status = "exploratory_best"

    x = result.x

    pair = [
        probe_from_angles(x[0], x[1]),
        probe_from_angles(x[2], x[3])
    ]

    noise_aware_probe_pairs[(ntype, sd, sp)] = pair

    restart_objectives = [
        r['objective_logdet']
        for r in noise_aware_restart_records
        if (
            r['noise_type'] == ntype
            and np.isclose(r['sigma_det'], sd)
            and np.isclose(r['sigma_phase'], sp)
        )
    ]

    noise_aware_probe_records.append({
        'noise_type': ntype,
        'sigma_det': sd,
        'sigma_phase': sp,
        'objective_logdet_opt': float(-result.fun),
        'opt_success': bool(result.success),
        'selection_status': selection_status,
        'nfev': int(result.nfev),
        'n_restarts': NOISE_AWARE_N_RESTARTS,
        'restart_objective_mean': float(np.mean(restart_objectives)),
        'restart_objective_std': (
            float(np.std(restart_objectives, ddof=1))
            if len(restart_objectives) > 1 else 0.0
        )
    })


noise_aware_probe_df = pd.DataFrame(noise_aware_probe_records)
noise_aware_restart_df = pd.DataFrame(noise_aware_restart_records)

display(noise_aware_probe_df)
display(noise_aware_restart_df)



# Final end-to-end evaluation

# All three designs are evaluated on matched latent parameter sets:
#   fixed baseline
#   nominal Fisher
#   noise-aware Fisher
#
# The final evaluation itself is not using the coarse optimization grid.

rng_noise_eval = np.random.default_rng(26099)

noise_eval_thetas = [
    sample_theta(
        rng_noise_eval,
        True,
        (.90, 1.10)
    )
    for _ in range(NOISE_AWARE_EVAL_N)
]


def eval_probe_pair_condition(
    probes,
    sd,
    sp,
    seed_base
):
    """
    Train/evaluate one probe design under one fixed noise condition.
    """

    Xtr, Ytr, _ = make_probe_condition_dataset_noise_v24(
        PAIR_TRAIN_N,
        seed_base + 1,
        probes,
        NPHOT,
        sd,
        sp,
        matched_train_thetas
    )

    Xte, Yte, _ = make_probe_condition_dataset_noise_v24(
        NOISE_AWARE_EVAL_N,
        seed_base + 2,
        probes,
        NPHOT,
        sd,
        sp,
        noise_eval_thetas
    )

    Htr = qml_ensemble_features_v24(
        Xtr,
        BEST_QML_SEEDS,
        .02
    )

    Hte = qml_ensemble_features_v24(
        Xte,
        BEST_QML_SEEDS,
        .02
    )

    mu, ss = fit_scaler_v24(Htr)

    Htr_s = apply_scaler_v24(Htr, mu, ss)
    Hte_s = apply_scaler_v24(Hte, mu, ss)

    model = Ridge(alpha=1e-2).fit(
        Htr_s,
        Ytr
    )

    pred = model.predict(Hte_s)

    return metrics_v24(Yte, pred)


noise_aware_eval_rows = []

for case_id, (ntype, sd, sp) in enumerate(NOISE_AWARE_CASES):

    designs = [
        (
            'fixed_baseline',
            fixed_probe_pair
        ),
        (
            'nominal_Fisher',
            optimized_probe_pair
        ),
        (
            'noise_aware_Fisher',
            noise_aware_probe_pairs[(ntype, sd, sp)]
        )
    ]

    for label, pair in designs:

        metrics = eval_probe_pair_condition(
            pair,
            sd,
            sp,
            27000 + 100 * case_id
        )

        noise_aware_eval_rows.append({
            'noise_type': ntype,
            'sigma_det': sd,
            'sigma_phase': sp,
            'probe_design': label,
            **metrics
        })


noise_aware_eval_df = pd.DataFrame(
    noise_aware_eval_rows
)

display(noise_aware_eval_df)




noise_aware_probe_df.to_csv(
    'V2_7_noise_aware_probe_design.csv',
    index=False
)

noise_aware_restart_df.to_csv(
    'V2_7_noise_aware_probe_restarts.csv',
    index=False
)

noise_aware_eval_df.to_csv(
    'V2_7_noise_aware_probe_evaluation.csv',
    index=False
)


print("\nNoise-aware Fisher experiment completed.")
print("Optimization kappa grid:", FISHER_KAPPA_GRID_OPT)
print("Final validation kappa grid:", FISHER_KAPPA_GRID_FULL)
print("Noise cases:", NOISE_AWARE_CASES)
print("Optimizer restarts:", NOISE_AWARE_N_RESTARTS)
print("DE maxiter:", NOISE_AWARE_MAXITER)
print("DE popsize:", NOISE_AWARE_POPSIZE)
print("Evaluation samples per condition:", NOISE_AWARE_EVAL_N)

In [ ]:


#   Matched QML/classical representation geometry



from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

X_rep,Y_rep,K_rep = domain_dataset_v24(1200, 28001)

Q_single = qrc_features(X_rep, BEST_QML_SEEDS[0], .02)
Q_ens = qml_ensemble_features_v24(X_rep, BEST_QML_SEEDS, .02)
C_rep = classical_features(X_rep, 29001)

# Standardize each representation before geometry analysis.
def representation_stats(F, y):
    Fs=StandardScaler().fit_transform(F)
    pca=PCA().fit(Fs)
    cum=np.cumsum(pca.explained_variance_ratio_)
    n95=int(np.searchsorted(cum,0.95)+1)
    Z=PCA(n_components=2).fit_transform(Fs)
    corr=float(np.corrcoef(Z[:,0],y[:,1])[0,1])
    bins=pd.qcut(y[:,1],q=3,labels=False,duplicates='drop')
    sil=float(silhouette_score(Z,bins))
    return {
        'n_features':int(F.shape[1]),
        'pc95_dimension':n95,
        'pc1_explained_variance':float(pca.explained_variance_ratio_[0]),
        'abs_corr_pc1_dEP':abs(corr),
        'silhouette_3regimes':sil
    }

representation_df=pd.DataFrame([
    {'representation':'classical_reservoir',**representation_stats(C_rep,Y_rep)},
    {'representation':'QML_single',**representation_stats(Q_single,Y_rep)},
    {'representation':'QML_ensemble_M4',**representation_stats(Q_ens,Y_rep)}
])
display(representation_df)
representation_df.to_csv('V2_6_representation_geometry_comparison.csv',index=False)


In [ ]:


#   Identical OOD comparison: QML ensemble vs classical



OOD_COMPARE_SEEDS=list(range(101,111))
ood_compare_rows=[]

for seed in OOD_COMPARE_SEEDS:
    Xtr,Ytr,Ktr=domain_dataset_v24(5000, 30000+seed)
    Xva,Yva,Kva=domain_dataset_v24(1500, 31000+seed)
    Xood,Yood,Kood=make_dataset(2000, 32000+seed, (1.22,1.38))

    #   Classical reservoir
    Rtr=classical_features(Xtr,seed)
    Rva=classical_features(Xva,seed)
    Rood=classical_features(Xood,seed)
    cm=Ridge(alpha=QML_ALPHA).fit(Rtr,Ytr)
    cp_va=cm.predict(Rva)
    cp_ood=cm.predict(Rood)

    #   QML ensemble
    Qtr=qml_ensemble_features_v24(Xtr,BEST_QML_SEEDS,.02)
    Qva=qml_ensemble_features_v24(Xva,BEST_QML_SEEDS,.02)
    Qood=qml_ensemble_features_v24(Xood,BEST_QML_SEEDS,.02)
    mu,ss=fit_scaler_v24(Qtr)
    qm=Ridge(alpha=1e-2).fit(apply_scaler_v24(Qtr,mu,ss),Ytr)
    qp_va=qm.predict(apply_scaler_v24(Qva,mu,ss))
    qp_ood=qm.predict(apply_scaler_v24(Qood,mu,ss))

    cmva=metrics_v24(Yva,cp_va); cmood=metrics_v24(Yood,cp_ood)
    qmva=metrics_v24(Yva,qp_va); qmood=metrics_v24(Yood,qp_ood)

    for model,mv,mo in [
        ('classical_reservoir',cmva,cmood),
        ('QML_ensemble_M4',qmva,qmood)
    ]:
        for metric in ['RMSE_eps','RMSE_d']:
            base=mv[metric]; out=mo[metric]
            row={
                'seed':seed,'model':model,'metric':metric,
                'ID':base,'OOD':out,
                'relative_degradation_pct':100*(out-base)/max(abs(base),1e-15)
            }
            ood_compare_rows.append(row)

ood_compare_df=pd.DataFrame(ood_compare_rows)
display(ood_compare_df.groupby(['model','metric'])[['ID','OOD','relative_degradation_pct']].agg(['mean','std']))
ood_compare_df.to_csv('V2_6_classical_vs_QML_OOD_comparison.csv',index=False)

# Paired seed-level differences in OOD performance (QML - classical).
ood_pivot=ood_compare_df.pivot_table(index=['seed','metric'],columns='model',values='OOD').reset_index()
for metric in ['RMSE_eps','RMSE_d']:
    g=ood_pivot[ood_pivot.metric.eq(metric)].copy()
    g['QML_minus_classical']=g['QML_ensemble_M4']-g['classical_reservoir']
    print(metric, 'mean OOD RMSE difference (QML - classical)=',g['QML_minus_classical'].mean())
    print(metric, 'paired differences=',g['QML_minus_classical'].to_numpy())


# Paired uncertainty analysis: every seed contributes one QML and one classical OOD score.
from scipy.stats import ttest_rel

def paired_bootstrap_mean_diff(d, B=20000, seed=86123):
    d=np.asarray(d,float)
    rng=np.random.default_rng(seed)
    means=np.empty(B)
    for b in range(B):
        means[b]=np.mean(rng.choice(d,size=d.size,replace=True))
    return float(np.mean(d)), tuple(np.quantile(means,[0.025,0.975]))

ood_stat_rows=[]
for metric in ['RMSE_eps','RMSE_d']:
    g=ood_pivot[ood_pivot.metric.eq(metric)].copy()
    d=g['QML_ensemble_M4'].to_numpy()-g['classical_reservoir'].to_numpy()
    md,ci=paired_bootstrap_mean_diff(d,seed=86123+(0 if metric=='RMSE_eps' else 1))
    tstat,tp=ttest_rel(g['QML_ensemble_M4'],g['classical_reservoir'])
    favorable=int(np.sum(d<0))
    # Relative degradation comparison: positive means QML degrades more.
    rel_q=ood_compare_df[(ood_compare_df.model=='QML_ensemble_M4') & (ood_compare_df.metric==metric)]['relative_degradation_pct'].to_numpy()
    rel_c=ood_compare_df[(ood_compare_df.model=='classical_reservoir') & (ood_compare_df.metric==metric)]['relative_degradation_pct'].to_numpy()
    rel_d=rel_q-rel_c
    rmd,rci=paired_bootstrap_mean_diff(rel_d,seed=86223+(0 if metric=='RMSE_eps' else 1))
    rt,rp=ttest_rel(rel_q,rel_c)
    ood_stat_rows.append({
        'metric':metric,'mean_QML_minus_classical':md,
        'bootstrap95_low':ci[0],'bootstrap95_high':ci[1],
        'paired_t_pvalue':float(tp),'seeds_QML_better_absolute':favorable,
        'mean_relative_degradation_QML_minus_classical_pct':rmd,
        'rel_bootstrap95_low':rci[0],'rel_bootstrap95_high':rci[1],
        'rel_paired_t_pvalue':float(rp),
        'interpretation':'absolute OOD comparison is favorable to QML' if (ci[1] < 0) else 'absolute OOD advantage is not statistically decisive',
        'robustness_interpretation':'QML degrades less under shift' if (rci[1] < 0) else ('QML degrades more under shift' if (rci[0] > 0) else 'relative OOD robustness is inconclusive')
    })
ood_stat_df=pd.DataFrame(ood_stat_rows)
display(ood_stat_df)
ood_stat_df.to_csv('V2_7_OOD_paired_uncertainty.csv',index=False)


In [ ]:


#   Equal-feature readout control



from sklearn.decomposition import PCA

FEATURE_BUDGETS=[32,64,120]
Xfb_tr,Yfb_tr,_=domain_dataset_v24(4000,33001)
Xfb_te,Yfb_te,_=make_dataset(1800,33002,(.80,1.20))

Rtr=classical_features(Xfb_tr,34001); Rte=classical_features(Xfb_te,34001)
Qtr=qrc_features(Xfb_tr,BEST_QML_SEEDS[0],.02); Qte=qrc_features(Xfb_te,BEST_QML_SEEDS[0],.02)
Etr=qml_ensemble_features_v24(Xfb_tr,BEST_QML_SEEDS,.02); Ete=qml_ensemble_features_v24(Xfb_te,BEST_QML_SEEDS,.02)

feature_budget_rows=[]
for name,Atr,Ate in [('classical_reservoir',Rtr,Rte),('QML_single',Qtr,Qte),('QML_ensemble_M4',Etr,Ete)]:
    mu,sd=fit_scaler_v24(Atr); Atr_s=apply_scaler_v24(Atr,mu,sd); Ate_s=apply_scaler_v24(Ate,mu,sd)
    max_budget=min(Atr_s.shape[1],max(FEATURE_BUDGETS))
    budgets=[b for b in FEATURE_BUDGETS if b<=max_budget]
    for b in budgets:
        if b<Atr_s.shape[1]:
            pca=PCA(n_components=b,random_state=1).fit(Atr_s)
            Ztr=pca.transform(Atr_s); Zte=pca.transform(Ate_s)
        else:
            Ztr,Zte=Atr_s,Ate_s
        m=Ridge(alpha=1e-2).fit(Ztr,Yfb_tr)
        pred=m.predict(Zte)
        feature_budget_rows.append({'model':name,'feature_budget':b,**metrics_v24(Yfb_te,pred)})

feature_budget_df=pd.DataFrame(feature_budget_rows)
display(feature_budget_df[['model','feature_budget','RMSE_eps','RMSE_d','R2_eps','R2_d']])
feature_budget_df.to_csv('V2_6_equal_feature_budget_comparison.csv',index=False)


In [ ]:

#   Multi-seed M=4 ensemble stability



ENSEMBLE_SETS = [
    [11,23,37,41],
    [13,29,43,47],
    [17,31,53,59],
    [19,37,61,71],
    [23,41,67,73],
]
Xstab_tr,Ystab_tr,_=domain_dataset_v24(4000,35001)
Xstab_te,Ystab_te,_=make_dataset(1800,35002,(1.22,1.38))

stability_rows=[]
for set_id,seeds in enumerate(ENSEMBLE_SETS):
    A=qml_ensemble_features_v24(Xstab_tr,seeds,.02)
    T=qml_ensemble_features_v24(Xstab_te,seeds,.02)
    mu,sd=fit_scaler_v24(A)
    mdl=Ridge(alpha=1e-2).fit(apply_scaler_v24(A,mu,sd),Ystab_tr)
    pred=mdl.predict(apply_scaler_v24(T,mu,sd))
    m=metrics_v24(Ystab_te,pred)
    stability_rows.append({'ensemble_set':set_id+1,'seeds':','.join(map(str,seeds)),**m})

ensemble_stability_df=pd.DataFrame(stability_rows)
ensemble_stability_summary=ensemble_stability_df[['RMSE_eps','RMSE_d','R2_eps','R2_d']].agg(['mean','std','min','max']).T
display(ensemble_stability_df)
display(ensemble_stability_summary)
ensemble_stability_df.to_csv('V2_7_ensemble_seed_stability.csv',index=False)




In [ ]:

#   FIGURE EXPORT


from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, linregress


# OUTPUT DIRECTORIES


RESULTS_DIR = Path("results")
FIG_DIR = RESULTS_DIR / "figures"
SUPP_DIR = FIG_DIR / "supplementary"
TABLE_DIR = RESULTS_DIR / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
SUPP_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig, filename):
    """
    Save publication-quality PDF and PNG.
    """
    pdf_path = FIG_DIR / f"{filename}.pdf"
    png_path = FIG_DIR / f"{filename}.png"

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    print(f"Saved: {pdf_path}")
    print(f"Saved: {png_path}")


print("=" * 72)
print("FINAL MANUSCRIPT FIGURE EXPORT")
print("=" * 72)



# MAIN BENCHMARK + ENSEMBLE SIZE


fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

#  Panel A: strict benchmark
bench_summary = (
    strict_benchmark
    .groupby("model")[
        [
            "RMSE_eps",
            "RMSE_d",
            "MAE_eps",
            "MAE_d",
            "R2_eps",
            "R2_d"
        ]
    ]
    .agg(["mean", "std"])
)

model_order = [
    "QML_reservoir",
    "classical_reservoir",
    "ridge"
]

x = np.arange(2)
width = 0.24

for j, model in enumerate(model_order):

    vals = [
        bench_summary.loc[model, ("RMSE_eps", "mean")],
        bench_summary.loc[model, ("RMSE_d", "mean")]
    ]

    errs = [
        bench_summary.loc[model, ("RMSE_eps", "std")],
        bench_summary.loc[model, ("RMSE_d", "std")]
    ]

    axes[0].bar(
        x + (j - 1) * width,
        vals,
        width,
        yerr=errs,
        capsize=4,
        label=model.replace("_", " ")
    )

axes[0].set_xticks(x)
axes[0].set_xticklabels(
    [r"$\epsilon$", r"$d_{\mathrm{EP}}$"]
)

axes[0].set_ylabel("RMSE")
axes[0].set_title("(a) Main benchmark")
axes[0].grid(
    axis="y",
    alpha=0.25
)

#  Panel B: M=1,2,4

ensemble_plot = v24_val.sort_values("M")

axes[1].plot(
    ensemble_plot["M"],
    ensemble_plot["RMSE_eps"],
    marker="o",
    linewidth=1.8,
    label=r"$\epsilon$"
)

axes[1].plot(
    ensemble_plot["M"],
    ensemble_plot["RMSE_d"],
    marker="s",
    linewidth=1.8,
    label=r"$d_{\mathrm{EP}}$"
)

axes[1].set_xticks(
    ensemble_plot["M"].tolist()
)

axes[1].set_yscale("log")
axes[1].set_xlabel("Ensemble size $M$")
axes[1].set_ylabel("Validation RMSE")
axes[1].set_title("(b) Ensemble-size dependence")
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

fig.tight_layout()

save_figure(
    fig,
    "fig02_benchmark_ensemble"
)

plt.show()
plt.close(fig)



# MATCHED-FEATURE CONTROL + CONTROLLED OOD


fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

 Panel A: matched-feature control
for model, g in feature_budget_df.groupby("model"):

    g = g.sort_values("feature_budget")

    axes[0].plot(
        g["feature_budget"],
        g["RMSE_eps"],
        marker="o",
        linewidth=1.8,
        label=model.replace("_", " ")
    )

axes[0].set_xlabel("Matched feature budget")
axes[0].set_ylabel(r"RMSE($\epsilon$)")
axes[0].set_title("(a) Matched-feature control")
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

#   Panel B: final controlled OOD

ood_mean = (
    ood_compare_df
    .groupby(["model", "metric"])["OOD"]
    .agg(["mean", "std"])
    .reset_index()
)

models = [
    "classical_reservoir",
    "QML_ensemble_M4"
]

model_labels = [
    "Classical reservoir",
    "QML ensemble"
]

x = np.arange(2)
width = 0.34

for j, metric in enumerate(
    ["RMSE_eps", "RMSE_d"]
):

    values = []
    errors = []

    for model in models:

        row = ood_mean[
            (ood_mean["model"] == model) &
            (ood_mean["metric"] == metric)
        ].iloc[0]

        values.append(row["mean"])
        errors.append(row["std"])

    axes[1].bar(
        x + (j - 0.5) * width,
        values,
        width,
        yerr=errors,
        capsize=4,
        label=(
            r"$\epsilon$"
            if metric == "RMSE_eps"
            else r"$d_{\mathrm{EP}}$"
        )
    )

axes[1].set_xticks(x)
axes[1].set_xticklabels(model_labels)
axes[1].set_ylabel("OOD RMSE")
axes[1].set_title("(b) Controlled unseen-device comparison")
axes[1].grid(
    axis="y",
    alpha=0.25
)
axes[1].legend(frameon=False)

fig.tight_layout()

save_figure(
    fig,
    "fig03_controls_ood"
)

plt.show()
plt.close(fig)



# DYNAMIC TRACKING + CLOSED-LOOP STABILIZATION


fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 4.8)
)

#  Panel A: dynamic d_EP

steps_per_traj = 120
traj_slice = slice(0, steps_per_traj)

time_axis = np.arange(
    len(dyn_test_Y)
)

axes[0].plot(
    time_axis[traj_slice],
    dyn_test_Y[traj_slice, 1],
    label=r"True $d_{\mathrm{EP}}$"
)

axes[0].plot(
    time_axis[traj_slice],
    dyn_pred[traj_slice, 1],
    "--",
    label=r"Estimated $d_{\mathrm{EP}}$"
)

axes[0].set_xlabel("Test trajectory sample")
axes[0].set_ylabel(r"$d_{\mathrm{EP}}$")
axes[0].set_title("(a) Dynamic EP tracking")
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

#  Panel B: dynamic epsilon

axes[1].plot(
    time_axis[traj_slice],
    dyn_test_Y[traj_slice, 0],
    label=r"True $\epsilon$"
)

axes[1].plot(
    time_axis[traj_slice],
    dyn_pred[traj_slice, 0],
    "--",
    label=r"Estimated $\epsilon$"
)

axes[1].set_xlabel("Test trajectory sample")
axes[1].set_ylabel(r"$\epsilon$")
axes[1].set_title("(b) Dynamic perturbation tracking")
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

#  Panel C: closed-loop aggregate result

controller_summary = (
    strict_controller
    .groupby("mode")[
        [
            "final_abs_d",
            "RMSE_d",
            "max_abs_d"
        ]
    ]
    .mean()
)

controller_order = [
    "none",
    "oracle",
    "qml"
]

controller_labels = [
    "Uncontrolled",
    "Oracle",
    "QML"
]

values = [
    controller_summary.loc[m, "RMSE_d"]
    for m in controller_order
]

axes[2].bar(
    controller_labels,
    values
)

axes[2].set_ylabel(r"RMSE($d_{\mathrm{EP}}$)")
axes[2].set_title("(c) Closed-loop stabilization")
axes[2].tick_params(
    axis="x",
    rotation=20
)
axes[2].grid(
    axis="y",
    alpha=0.25
)

fig.tight_layout()

save_figure(
    fig,
    "fig04_tracking_control"
)

plt.show()
plt.close(fig)



# FISHER INFORMATION + END-TO-END VALIDATION


fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 4.8)
)

fd = probe_design_df.set_index("design")

#  Panel A: effective Fisher information

axes[0].bar(
    ["Fixed", "Fisher-D"],
    [
        fd.loc[
            "fixed_baseline",
            "mean_logdet_effective_Fisher"
        ],
        fd.loc[
            "Fisher_D_optimal",
            "mean_logdet_effective_Fisher"
        ]
    ]
)

axes[0].set_ylabel(
    r"Mean $\log\det F_{\mathrm{eff}}$"
)
axes[0].set_title(
    "(a) Effective Fisher information"
)
axes[0].grid(
    axis="y",
    alpha=0.25
)

#  Panel B: Fisher local uncertainty

axes[1].bar(
    np.arange(2) - 0.17,
    [
        fd.loc[
            "fixed_baseline",
            "mean_CRLB_std_eps"
        ],
        fd.loc[
            "Fisher_D_optimal",
            "mean_CRLB_std_eps"
        ]
    ],
    width=0.34,
    label=r"$\epsilon$"
)

axes[1].bar(
    np.arange(2) + 0.17,
    [
        fd.loc[
            "fixed_baseline",
            "mean_CRLB_std_d"
        ],
        fd.loc[
            "Fisher_D_optimal",
            "mean_CRLB_std_d"
        ]
    ],
    width=0.34,
    label=r"$d_{\mathrm{EP}}$"
)

axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(
    ["Fixed", "Fisher-D"]
)

axes[1].set_ylabel(
    "Local Fisher-based uncertainty"
)

axes[1].set_title(
    "(b) Local Fisher uncertainty"
)

axes[1].grid(
    axis="y",
    alpha=0.25
)

axes[1].legend(frameon=False)

#  Panel C: end-to-end RMSE

e2e = probe_e2e_df.set_index(
    "probe_design"
)

axes[2].bar(
    np.arange(2) - 0.17,
    [
        e2e.loc["fixed_baseline", "RMSE_eps"],
        e2e.loc["Fisher_D_optimal", "RMSE_eps"]
    ],
    width=0.34,
    label=r"$\epsilon$"
)

axes[2].bar(
    np.arange(2) + 0.17,
    [
        e2e.loc["fixed_baseline", "RMSE_d"],
        e2e.loc["Fisher_D_optimal", "RMSE_d"]
    ],
    width=0.34,
    label=r"$d_{\mathrm{EP}}$"
)

axes[2].set_xticks([0, 1])
axes[2].set_xticklabels(
    ["Fixed", "Fisher-D"]
)

axes[2].set_ylabel("RMSE")
axes[2].set_title(
    "(c) End-to-end sensing"
)

axes[2].grid(
    axis="y",
    alpha=0.25
)

axes[2].legend(frameon=False)

fig.tight_layout()

save_figure(
    fig,
    "fig05_fisher_design"
)

plt.show()
plt.close(fig)



# PHOTON-BUDGET ROBUSTNESS


fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 4.8)
)

for label, g in probe_budget_df.groupby(
    "probe_design"
):

    g = g.sort_values("Nphot")

    display_label = label.replace(
        "_", " "
    )

    axes[0].plot(
        g["Nphot"],
        g["RMSE_eps"],
        marker="o",
        linewidth=1.8,
        label=display_label
    )

    axes[1].plot(
        g["Nphot"],
        g["RMSE_d"],
        marker="s",
        linewidth=1.8,
        label=display_label
    )

axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel(
    "Photons per population channel"
)
axes[0].set_ylabel(
    r"RMSE($\epsilon$)"
)
axes[0].set_title(
    "(a) Photon-budget dependence: $\epsilon$"
)
axes[0].grid(alpha=0.25)

axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel(
    "Photons per population channel"
)
axes[1].set_ylabel(
    r"RMSE($d_{\mathrm{EP}}$)"
)
axes[1].set_title(
    "(b) Photon-budget dependence: $d_{\mathrm{EP}}$"
)
axes[1].grid(alpha=0.25)

axes[0].legend(frameon=False)

fig.tight_layout()

save_figure(
    fig,
    "fig06_photon_budget"
)

plt.show()
plt.close(fig)





fig, axes = plt.subplots(
    2,
    2,
    figsize=(11, 7.5),
    constrained_layout=True
)

noise_panels = [
    (
        "detector",
        "RMSE_eps",
        r"$\sigma_{\mathrm{det}}$",
        r"RMSE($\epsilon$)",
        r"(a) Detector noise: $\epsilon$"
    ),
    (
        "detector",
        "RMSE_d",
        r"$\sigma_{\mathrm{det}}$",
        r"RMSE($d_{\mathrm{EP}}$)",
        r"(b) Detector noise: $d_{\mathrm{EP}}$"
    ),
    (
        "phase",
        "RMSE_eps",
        r"$\sigma_{\mathrm{phase}}$",
        r"RMSE($\epsilon$)",
        r"(c) Phase noise: $\epsilon$"
    ),
    (
        "phase",
        "RMSE_d",
        r"$\sigma_{\mathrm{phase}}$",
        r"RMSE($d_{\mathrm{EP}}$)",
        r"(d) Phase noise: $d_{\mathrm{EP}}$"
    )
]

for ax, (
    noise_type,
    metric,
    xlabel,
    ylabel,
    title
) in zip(
    axes.ravel(),
    noise_panels
):

    g = (
        noise_matched_gain_df[
            noise_matched_gain_df["noise_type"] == noise_type
        ]
        .sort_values("noise_level")
    )

    x = g["noise_level"].to_numpy()

    fixed_y = g[
        f"{metric}_fixed"
    ].to_numpy()

    fisher_y = g[
        f"{metric}_fisher"
    ].to_numpy()

    ax.plot(
        x,
        fixed_y,
        marker="o",
        linewidth=1.8,
        markersize=5,
        label="Fixed probes"
    )

    ax.plot(
        x,
        fisher_y,
        marker="s",
        linewidth=1.8,
        markersize=5,
        label="Fisher-D-optimal"
    )

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(
        True,
        alpha=0.25
    )

    ax.set_xscale(
        "symlog",
        linthresh=(
            0.001
            if noise_type == "detector"
            else 0.005
        )
    )

axes[0, 0].legend(
    frameon=False,
    loc="best"
)

fig.suptitle(
    "Measurement-noise robustness",
    fontsize=13
)

save_figure(
    fig,
    "fig07_noise_robustness"
)

plt.show()
plt.close(fig)



# IDENTIFIABILITY + REPRESENTATION

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 4.8)
)

#  Panel A: identifiability

baseline_rank = float(
    Fbase["rank"].mean()
)

phase_rank = float(
    Fphase["rank"].mean()
)

axes[0].bar(
    ["Without\nphase", "With\nphase"],
    [
        baseline_rank,
        phase_rank
    ]
)

axes[0].set_ylabel(
    "Mean measurement-Jacobian rank"
)

axes[0].set_title(
    "(a) Phase-reference identifiability"
)

axes[0].set_ylim(
    0,
    max(5.5, phase_rank + 0.5)
)

axes[0].grid(
    axis="y",
    alpha=0.25
)

#  Panel B: PCA explained variance

components = np.arange(
    1,
    len(pca.explained_variance_ratio_) + 1
)

axes[1].bar(
    components,
    pca.explained_variance_ratio_ * 100
)

axes[1].set_xlabel(
    "Principal component"
)

axes[1].set_ylabel(
    "Explained variance (%)"
)

axes[1].set_title(
    "(b) Reservoir representation"
)

axes[1].grid(
    axis="y",
    alpha=0.25
)

#  Panel C: PC1 vs d_EP

rho, pval = spearmanr(
    H_pca[:, 0],
    Yvis[:, 1]
)

axes[2].scatter(
    H_pca[:, 0],
    Yvis[:, 1],
    s=12,
    alpha=0.55
)

coef = np.polyfit(
    H_pca[:, 0],
    Yvis[:, 1],
    1
)

xx = np.linspace(
    H_pca[:, 0].min(),
    H_pca[:, 0].max(),
    200
)

axes[2].plot(
    xx,
    coef[0] * xx + coef[1],
    "--"
)

axes[2].set_xlabel(
    "Reservoir PC1"
)

axes[2].set_ylabel(
    r"$d_{\mathrm{EP}}$"
)

axes[2].set_title(
    fr"(c) PC1 vs $d_{{\mathrm{{EP}}}}$, "
    fr"$\rho={rho:.4f}$"
)

axes[2].grid(
    alpha=0.25
)

fig.tight_layout()

save_figure(
    fig,
    "fig08_identifiability_representation"
)

plt.show()
plt.close(fig)



# FINITE-RANGE SCALABILITY


fig, ax = plt.subplots(
    figsize=(8, 5)
)

D = scalability_df[
    "system_dimension_D"
].to_numpy()

reservoir_time = scalability_df[
    "reservoir_total_time_s"
].to_numpy()

spectral_time = scalability_df[
    "spectral_recalibration_time_s"
].to_numpy()

ax.loglog(
    D,
    reservoir_time,
    marker="o",
    linewidth=1.8,
    label="Reservoir route"
)

ax.loglog(
    D,
    spectral_time,
    marker="s",
    linewidth=1.8,
    label="Repeated spectral scans"
)


valid = D > 2

reservoir_slope = linregress(
    np.log(D[valid]),
    np.log(
        scalability_df.loc[
            valid,
            "reservoir_physics_time_s"
        ]
    )
).slope

spectral_slope = linregress(
    np.log(D[valid]),
    np.log(
        scalability_df.loc[
            valid,
            "spectral_recalibration_time_s"
        ]
    )
).slope

ax.set_xlabel(
    "System dimension $D$"
)

ax.set_ylabel(
    "Wall-clock time (s)"
)

ax.set_title(
    "Finite-range computational scaling"
)

ax.grid(
    alpha=0.25
)

ax.legend(
    frameon=False
)

ax.text(
    0.05,
    0.05,
    (
        f"Reservoir physics slope = {reservoir_slope:.2f}\n"
        f"Spectral slope = {spectral_slope:.2f}"
    ),
    transform=ax.transAxes
)

fig.tight_layout()

save_figure(
    fig,
    "fig09_scalability"
)

plt.show()
plt.close(fig)



# FIVE-SEED M=4 STABILITY


fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4.5)
)

x = ensemble_stability_df[
    "ensemble_set"
].to_numpy()

axes[0].plot(
    x,
    ensemble_stability_df["RMSE_eps"],
    marker="o"
)

axes[0].set_xlabel(
    "Ensemble seed-set"
)

axes[0].set_ylabel(
    r"RMSE($\epsilon$)"
)

axes[0].set_title(
    "Seed sensitivity: $\epsilon$"
)

axes[0].grid(alpha=0.25)

axes[1].plot(
    x,
    ensemble_stability_df["RMSE_d"],
    marker="s"
)

axes[1].set_xlabel(
    "Ensemble seed-set"
)

axes[1].set_ylabel(
    r"RMSE($d_{\mathrm{EP}}$)"
)

axes[1].set_title(
    "Seed sensitivity: $d_{\mathrm{EP}}$"
)

axes[1].grid(alpha=0.25)

fig.tight_layout()

# Save into supplementary folder manually.
fig.savefig(
    SUPP_DIR / "figS1_seed_stability.pdf",
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    SUPP_DIR / "figS1_seed_stability.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)



# BIAS BEFORE / DURING / AFTER CONTROL


fig, ax = plt.subplots(
    figsize=(7, 4.5)
)

phase_order = [
    "before",
    "during",
    "after"
]

bias_plot = (
    strict_bias
    .set_index("phase")
    .reindex(phase_order)
)

mean_bias = bias_plot[
    "mean_bias"
].to_numpy()

lower = (
    mean_bias -
    bias_plot["CI_low"].to_numpy()
)

upper = (
    bias_plot["CI_high"].to_numpy() -
    mean_bias
)

ax.errorbar(
    phase_order,
    mean_bias,
    yerr=[
        lower,
        upper
    ],
    fmt="o",
    capsize=4
)

ax.axhline(
    0,
    linestyle="--"
)

ax.set_xlabel(
    "Restoration phase"
)

ax.set_ylabel(
    r"Bias $b_{\epsilon}$"
)

ax.set_title(
    "Perturbation-estimation bias"
)

ax.grid(alpha=0.25)

fig.tight_layout()

fig.savefig(
    SUPP_DIR / "figS2_bias_restoration.pdf",
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    SUPP_DIR / "figS2_bias_restoration.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)



# DYNAMIC TRACKING ERROR DISTRIBUTIONS


err_eps = (
    dyn_pred[:, 0] -
    dyn_test_Y[:, 0]
)

err_d = (
    dyn_pred[:, 1] -
    dyn_test_Y[:, 1]
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4)
)

axes[0].hist(
    err_eps,
    bins=35
)

axes[0].axvline(
    0,
    linestyle="--"
)

axes[0].set_xlabel(
    r"$\hat{\epsilon}-\epsilon$"
)

axes[0].set_ylabel(
    "Count"
)

axes[0].set_title(
    r"Dynamic tracking error: $\epsilon$"
)

axes[1].hist(
    err_d,
    bins=35
)

axes[1].axvline(
    0,
    linestyle="--"
)

axes[1].set_xlabel(
    r"$\hat d_{\mathrm{EP}}-d_{\mathrm{EP}}$"
)

axes[1].set_ylabel(
    "Count"
)

axes[1].set_title(
    r"Dynamic tracking error: $d_{\mathrm{EP}}$"
)

fig.tight_layout()

fig.savefig(
    SUPP_DIR / "figS3_dynamic_errors.pdf",
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    SUPP_DIR / "figS3_dynamic_errors.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)



# EXPORT SUPPORTING RESULT TABLES


tables_to_save = {
    "ensemble_size_v24.csv": v24_val,
    "matched_feature_control.csv": feature_budget_df,
    "controlled_OOD_comparison.csv": ood_compare_df,
    "controlled_OOD_statistics.csv": ood_stat_df,
    "Fisher_probe_design.csv": probe_design_df,
    "Fisher_end_to_end.csv": probe_e2e_df,
    "Fisher_photon_budget.csv": probe_budget_df,
    "noise_matched_gain.csv": noise_matched_gain_df,
    "ensemble_seed_stability.csv": ensemble_stability_df,
    "sensing_bias.csv": strict_bias,
    "representation_PCA.csv": pca_geometry_df,
    "scalability.csv": scalability_df
}

for filename, df in tables_to_save.items():

    df.to_csv(
        TABLE_DIR / filename,
        index=False
    )

    print(
        f"Saved table: "
        f"{TABLE_DIR / filename}"
    )



# FINAL REPORT


print("\n" + "=" * 72)
print("FINAL MANUSCRIPT EXPORT COMPLETE")
print("=" * 72)

print("\nMain figures:")
for p in sorted(FIG_DIR.glob("fig*.pdf")):
    print("  ", p)

print("\nSupplementary figures:")
for p in sorted(SUPP_DIR.glob("*.pdf")):
    print("  ", p)

print("\nTables:")
for p in sorted(TABLE_DIR.glob("*.csv")):
    print("  ", p)

#print("\nNo model training or expensive experiment was rerun.")